In [ ]:
from pathlib import Path
import importlib
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.backends.backend_pdf import PdfPages

import pmt.io as pmt_io
pmt_io = importlib.reload(pmt_io)  # pick up local helper changes in an existing kernel
import pmt.selection as pmt_selection
pmt_selection = importlib.reload(pmt_selection)
import pmt.testing_summary as pmt_testing_summary
pmt_testing_summary = importlib.reload(pmt_testing_summary)
load_baseline_reference = pmt_io.load_baseline_reference
load_event_waveforms = pmt_io.load_event_waveforms
load_acquisition_timing_exposure = (
    pmt_io.load_acquisition_timing_exposure
)
stream_event_qdc_diagnostics = pmt_io.stream_event_qdc_diagnostics
stream_event_charge_method_comparison = (
    pmt_io.stream_event_charge_method_comparison
)
resolve_cached_event_files = pmt_io.resolve_cached_event_files
save_shape_diagnostics_cache = (
    pmt_testing_summary.save_shape_diagnostics_cache
)
generate_shape_diagnostics_pdf_from_cache = (
    pmt_testing_summary.generate_shape_diagnostics_pdf_from_cache
)
add_learned_signal_classification = pmt_selection.add_learned_signal_classification
from pmt.provenance import load_selection_cache_provenance

# Release large objects and an interrupted PDF handle from a previous run.
_previous_testing_pdf = globals().get("_testing_pdf_pages")
if _previous_testing_pdf is not None:
    try:
        _previous_testing_pdf.close()
    except Exception:
        pass
_previous_testing_pdf_temporary_path = globals().get(
    "_testing_pdf_temporary_path"
)
if _previous_testing_pdf_temporary_path is not None:
    _previous_testing_pdf_temporary_path = Path(
        _previous_testing_pdf_temporary_path
    )
    if _previous_testing_pdf_temporary_path.is_file():
        _previous_testing_pdf_temporary_path.unlink()
_previous_testing_show = globals().get("_testing_original_show")
if callable(_previous_testing_show):
    plt.show = _previous_testing_show
for _large_name in (
    "sample_df", "finite_rms_events", "pedestal_candidates",
    "peak_candidates", "signal_noise_like_candidates",
    "signal_off_time_candidates", "signal_oscillatory_candidates",
    "small_additional_peak_candidates", "small_additional_peak_blue_candidates",
    "generic_bad_shape_candidates",
    "multi_peak_candidates", "wide_peak_candidates",
    "sampled_pedestals", "sampled_peaks", "sampled_multi_peaks",
    "sampled_oscillatory", "sampled_small_additional_peaks",
    "sampled_small_additional_peak_blue",
    "sampled_generic_bad_shape", "sampled_events",
    "sampled_events_post_mismatch_cut",
    "sampled_peaks_before_mismatch_cut",
    "shape_cache_signal_events_before_mismatch_cut",
    "peak_qdc_values_before_mismatch_cut",
    "peak_raw_full_waveform_qdc_values_before_mismatch_cut",
    "peak_max_amplitudes_mV_before_mismatch_cut",
    "sampled_peak_positions", "sampled_pedestal_positions",
    "sampled_peak_positions_after_mismatch_cut",
    "noisiest_kept_events", "noisiest_waveforms_mV",
    "streamed_qdc", "pedestal_qdc_values",
    "pedestal_full_waveform_qdc_values", "peak_qdc_values",
    "pedestal_raw_full_waveform_qdc_values",
    "peak_raw_full_waveform_qdc_values",
    "combined_qdc_values", "overlap_diagnostic_events",
    "overlap_diagnostic_waveforms_mV",
    "high_qdc_diagnostic_events", "high_qdc_diagnostic_waveforms_mV",
    "shape_quality_events", "shape_quality_results",
    "signal_template_reference_records", "signal_shape_template",
    "shape_template_candidates", "shape_reference_waveforms_mV",
    "shape_reference_shapes",
    "shape_cache_signal_events", "shape_cache_pedestal_events",
    "qdc_separation_method_populations", "qdc_separation_results",
):
    globals().pop(_large_name, None)
gc.collect()

# #################### DATASET AND PDF OUTPUT ####################
# Choose the cached acquisition. Set the PDF path to None for inline plots only.
sample_acquisition = globals().get(
    "testing_sample_acquisition_override",
    "WA0089_775V_Dark_20microV_trig",
)
sample_cache_path = Path(globals().get(
    "testing_sample_cache_path_override",
    Path("plots/dark_counts/fit_data/20_8") / f"{sample_acquisition}_df.pkl",
))
testing_summary_pdf_path = (
    Path("plots/dark_counts_20_8")
    / f"{sample_acquisition}_testing_summary.pdf"
)  # e.g. Path("plots/dark_counts_16_8") / f"{sample_acquisition}_testing_summary.pdf"
shape_diagnostics_pdf_path = Path(globals().get(
    "testing_shape_diagnostics_pdf_path_override",
    Path("plots/dark_counts_20_8")
    / f"{sample_acquisition}_waveform_shape_diagnostics.pdf",
))
shape_metrics_cache_path = (
    shape_diagnostics_pdf_path.parent / "shape_metric_cache"
    / f"{sample_acquisition}_shape_metrics.pkl"
)
# #################################################################

# #################### EVENT SAMPLING AND RAM #####################
# Maximum events used for QDC calculations; raw waveforms are streamed in chunks.
max_pedestal_waveforms_for_qdc = 300_000  # use at most this many; no repetition
max_peak_waveforms_for_qdc = 300_000      # use at most this many; no repetition
qdc_read_chunk_size = 128  # bounds raw waveform RAM during the sequential pass
# Only these small subsets are retained for the main waveform overlay.
n_pedestal_waveforms_to_plot = 10  # QDC still uses all n_pedestal_waveforms
n_peak_waveforms_to_plot = 10      # QDC still uses all n_peak_waveforms
n_noisiest_kept_waveforms_to_plot = 10
# Reproducible sampling and filters applied to the blue signal population.
sample_seed = 12345
sample_peak_snr_min = 0.0
sample_peak_amplitude_max_mV = None  # None -> highest finite excursion in this dataset
# #################################################################

# #################### HISTOGRAMS AND SCATTERS ####################
# Display resolution and density/rate choice; plotting caps do not reduce calculations.
qdc_histogram_bins = 40
qdc_histogram_density = False
qdc_use_timing_rate_units = True  # auto-use a matching *_timing.csv
qdc_timing_time_column = "Acquisition_s"  # live time; Batch_s includes deadtime
amplitude_histogram_bins = 60
amplitude_qdc_scatter_max_points = 50_000  # plotting cap only; calculations use all
amplitude_qdc_density_bins = 100  # bins per axis for peak-population density colors
rise_time_histogram_bins = 60
rise_time_qdc_density_bins = 100  # bins per axis for the blue-population 2D distribution
# #################################################################

# #################### AUTOMATIC QDC OVERLAP ######################
# Histogram intersection defines the overlap; central shared mass defines its interval.
qdc_overlap_histogram_bins = 120
qdc_overlap_display_quantiles = (0.001, 0.999)
qdc_overlap_mass_quantiles = (0.05, 0.95)
# Extend only the signal diagnostic range above the overlap by this width fraction.
qdc_overlap_signal_upper_extension_fraction = 0.25
qdc_overlap_diagnostic_max_waveforms_per_population = 10
# Locate the smoothed valley between the pedestal and signal modes for each
# integration method. Class balancing prevents the much larger pedestal
# population from moving the separator toward the signal mode.
qdc_separator_histogram_bins = 160
qdc_separator_smoothing_sigma_bins = 2.0
qdc_separator_range_quantiles = (0.001, 0.999)
qdc_separator_balance_classes = True
# #################################################################

# #################### HIGH-QDC TAIL DIAGNOSTICS ##################
# Sample representatives from each population's top tail; expand it to contain >=10.
qdc_high_tail_fraction = 0.05
qdc_high_tail_min_candidates_per_population = 10
qdc_high_tail_waveforms_to_plot_per_population = 10
# #################################################################

# #################### PEDESTAL CHARGE DIAGNOSTICS ################
# QDC range/count for suspicious pedestal examples, tail report, and full-trace baseline.
pedestal_qdc_diagnostic_range_mV_ns = (4.0, 10.0)
pedestal_qdc_diagnostic_max_waveforms = 8
pedestal_qdc_tail_threshold_mV_ns = 4.0  # report fraction above this QDC
pedestal_full_waveform_baseline_sidebands_ns = ((0.0, 30.0), (70.0, 100.0))
# #################################################################

# #################### SPECIAL WAVEFORM EXAMPLES ##################
# Maximum examples retained for each peak multiplicity/shape diagnostic figure.
multi_peak_diagnostic_max_waveforms = 10
oscillatory_diagnostic_max_waveforms = 10
small_additional_peak_diagnostic_max_waveforms = 10
small_additional_peak_blue_diagnostic_max_waveforms = 10
generic_bad_shape_diagnostic_max_waveforms = 10
peak_low_qdc_diagnostic_range_mV_ns = (0, 3)
peak_low_qdc_diagnostic_max_waveforms = 10
wide_peak_diagnostic_min_width_ns = 20.0
wide_peak_diagnostic_max_waveforms = 10
# #################################################################

# #################### SHAPE-METRIC DIAGNOSTICS ###################
# Testing-only signal mismatch cut; it does not rewrite Selection classes.
# Pedestal growth/local-RMS metrics remain diagnostic-only and are not applied.
shape_template_reference_snr = 15.0
shape_template_reference_waveforms = 200  # bounded RAM for median template
shape_template_min_reference_waveforms = 20
shape_template_relative_window_ns = (-20.0, 40.0)
shape_metric_histogram_bins = 60
shape_metric_qdc_bins = 100
# Ten 10 ns blocks for a 100 ns trace; compare the median robust RMS of
# the first two blocks with the last two to expose gradual noise growth.
shape_pedestal_time_blocks = 10
shape_pedestal_edge_blocks = 2
# Local robust-RMS constancy: (90th - 10th percentile) / median RMS.
shape_pedestal_local_rms_window_ns = 4.0
shape_pedestal_rms_variation_max = 0.20
shape_worst_waveforms_to_plot = 10
shape_signal_reject_upper_fraction = 0.05  # reject highest mismatch fraction
shape_pedestal_growth_ratio_max = 2.0
# #################################################################

# #################### LEGACY-CACHE FALLBACK ######################
# Used only when an old cache lacks current generic-shape/dominance classifications.
testing_generic_shape_ranges = {
    "peak_width_ns": (6.0, 18.0),
    "rise_time_10_90_ns": (2.0, 10.0),
    "fall_time_90_10_ns": (7.0, 38.0),
}
testing_max_additional_peak_fraction = 0.75
# #################################################################

# Optionally mirror every inline figure into one multipage PDF.
_testing_pdf_pages = None
_testing_pdf_temporary_path = None
_testing_pdf_page_count = 0
_testing_pdf_saved_figure_ids = set()
_testing_original_show = getattr(
    plt.show, "_testing_original_show", plt.show
)
if testing_summary_pdf_path is not None:
    testing_summary_pdf_path = Path(testing_summary_pdf_path)
    testing_summary_pdf_path.parent.mkdir(parents=True, exist_ok=True)
    _testing_pdf_temporary_path = testing_summary_pdf_path.with_suffix(
        testing_summary_pdf_path.suffix + ".partial"
    )
    if _testing_pdf_temporary_path.is_file():
        _testing_pdf_temporary_path.unlink()

    def _testing_pdf_show(*args, **kwargs):
        global _testing_pdf_pages, _testing_pdf_page_count
        if _testing_pdf_pages is None:
            _testing_pdf_pages = PdfPages(_testing_pdf_temporary_path)
        figures_to_release = [
            plt.figure(figure_number)
            for figure_number in tuple(plt.get_fignums())
        ]
        for figure in figures_to_release:
            if id(figure) not in _testing_pdf_saved_figure_ids:
                _testing_pdf_pages.savefig(figure, bbox_inches="tight")
                _testing_pdf_saved_figure_ids.add(id(figure))
                _testing_pdf_page_count += 1
        _testing_original_show(*args, **kwargs)
        # Inline output has already been rendered and the PDF page saved.
        # Clear artists/large image buffers so dozens of figures do not stay
        # resident until this long cell finishes.
        for figure in figures_to_release:
            figure.clear()
            plt.close(figure)

    _testing_pdf_show._testing_original_show = _testing_original_show
    plt.show = _testing_pdf_show

# Load the exact preprocessing and learned fixed-window settings used by Fit.
print(f"Starting testing diagnostics from {sample_cache_path} ...", flush=True)
sample_df = pd.read_pickle(sample_cache_path)
if sample_peak_amplitude_max_mV is None:
    finite_excursions_mV = sample_df["max_excursion_amplitude_mV"].loc[
        np.isfinite(sample_df["max_excursion_amplitude_mV"])
    ]
    if finite_excursions_mV.empty:
        raise ValueError("Dataset has no finite maximum-excursion amplitudes")
    sample_peak_amplitude_max_mV = float(finite_excursions_mV.max())
    print(
        "Peak-amplitude sampling cutoff set to dataset maximum excursion: "
        f"{sample_peak_amplitude_max_mV:.6g} mV"
    )
sample_manifest = load_selection_cache_provenance(
    sample_df, expected_acquisition=sample_acquisition
)
sample_preprocessing = sample_manifest["preprocessing"]
cached_signal_classification = sample_manifest["selection"].get(
    "learned_signal_classification", {}
)
current_classification_columns = {
    "has_generic_pulse_shape", "signal_bad_generic_shape",
    "has_dominant_peak", "signal_with_small_additional_peaks",
    "largest_additional_signal_like_peak_fraction",
}
recomputed_legacy_classification = not current_classification_columns.issubset(
    sample_df.columns
)
if recomputed_legacy_classification:
    if "n_signal_like_peaks" not in sample_df.columns:
        raise ValueError(
            "This cache predates the pre-width candidate count and cannot be "
            "classified in memory; rerun Selection.ipynb."
        )
    missing_candidate_ratio = (
        "largest_additional_signal_like_peak_fraction" not in sample_df.columns
    )
    if missing_candidate_ratio:
        sample_df = sample_df.copy()
        sample_df["largest_additional_signal_like_peak_fraction"] = np.where(
            sample_df["n_signal_like_peaks"].le(1), 0.0, np.inf
        )
        print(
            "Legacy cache has no additional/main peak-amplitude ratio. "
            "Multiple-candidate events remain conservatively rejected; rerun "
            "Selection to recover real pulses with small ringing."
        )
    print(
        "Recomputing current classification flags in memory. The cache is not modified."
    )
    sample_df, sample_signal_classification = add_learned_signal_classification(
        sample_df,
        reference_snr=cached_signal_classification.get("reference_snr", 15.0),
        shape_quantiles=cached_signal_classification.get(
            "shape_quantiles", (0.01, 0.99)
        ),
        timing_quantiles=cached_signal_classification.get(
            "timing_quantiles", (0.01, 0.99)
        ),
        min_reference_pulses=cached_signal_classification.get(
            "min_reference_pulses", 100
        ),
        generic_shape_ranges=testing_generic_shape_ranges,
        max_additional_peak_fraction=(
            cached_signal_classification.get(
                "max_additional_peak_fraction",
                testing_max_additional_peak_fraction,
            )
        ),
    )
else:
    sample_signal_classification = cached_signal_classification
sample_generic_shape_ranges = sample_signal_classification.get(
    "generic_shape_ranges", testing_generic_shape_ranges
)
sample_max_additional_peak_fraction = sample_signal_classification.get(
    "max_additional_peak_fraction", testing_max_additional_peak_fraction
)
print(
    "Maximum additional/main candidate amplitude fraction: "
    f"{sample_max_additional_peak_fraction:g}"
)
print(f"Generic pulse-shape bounds: {sample_generic_shape_ranges}")
sample_baseline_rms_cut = sample_preprocessing.get("baseline_rms_quantile_cut")
if sample_baseline_rms_cut is None:
    print("This cache predates the global baseline-RMS quantile-cut metadata; continuing without an RMS-cut summary.")
elif not sample_baseline_rms_cut.get("enabled", False):
    kept_after_saturation = sample_baseline_rms_cut.get("kept_after_saturation")
    kept_note = (
        f" ({kept_after_saturation:,} events retained after saturation)"
        if kept_after_saturation is not None else ""
    )
    print(f"Baseline RMS cut: disabled; no waveforms were rejected by baseline RMS{kept_note}.")
else:
    print(
        f"Baseline RMS cut: q={sample_baseline_rms_cut['quantile']:g}, "
        f"threshold={sample_baseline_rms_cut['threshold_mV']:.6g} mV, "
        f"kept {sample_baseline_rms_cut['kept_after_rms_cut']:,}/"
        f"{sample_baseline_rms_cut['kept_after_saturation']:,} after saturation"
    )
sample_fixed_window = sample_preprocessing["fixed_pulse_window_charge"]
if not sample_fixed_window.get("enabled", False):
    raise ValueError("The selected cache has no learned fixed pulse window")
if recomputed_legacy_classification and not sample_fixed_window.get(
    "generic_shape_ranges"
):
    print(
        "Legacy-cache note: its stored fixed charge window was learned before "
        "generic shape cuts. Classification is updated in memory, but the stored "
        "integration window is unchanged."
    )
sample_fixed_window_ns = tuple(sample_fixed_window["window_ns"])
sample_baseline_window_ns = tuple(sample_preprocessing["baseline_window_ns"])
required_signal_classification_columns = {
    "has_pulse_shape", "is_trigger_aligned", "signal_good",
    "has_isolated_peak", "has_dominant_peak", "signal_oscillatory",
    "signal_with_small_additional_peaks",
    "has_generic_pulse_shape", "signal_bad_generic_shape",
    "signal_noise_like", "signal_off_time",
}
missing_signal_classification_columns = (
    required_signal_classification_columns.difference(sample_df.columns)
)
if missing_signal_classification_columns:
    raise ValueError(
        f"Cache is missing learned signal classification columns "
        f"{sorted(missing_signal_classification_columns)}; rerun Selection.ipynb."
    )

sample_fixed_charge_column = sample_fixed_window.get(
    "column", "charge_fixed_pulse_window_mV_ns"
)
sample_fixed_charge_available = sample_fixed_charge_column in sample_df.columns
sampled_event_columns = [
    column for column in dict.fromkeys([
        "event_file", "event_segment", "n_peaks",
        "baseline_rms_mV", "peak_time_ns", "peak_width_ns",
        "rise_time_10_90_ns", "fall_time_90_10_ns",
        "peak_amplitude_mV", "max_excursion_amplitude_mV", "snr",
        "has_pulse_shape", "is_trigger_aligned", "signal_good",
        "has_isolated_peak", "has_dominant_peak",
        "signal_oscillatory", "signal_with_small_additional_peaks",
        "has_generic_pulse_shape", "signal_bad_generic_shape",
        "signal_like_bad_shape", "signal_noise_like",
        "signal_off_time",
        "largest_additional_signal_like_peak_fraction",
        sample_fixed_charge_column,
    ]) if column in sample_df.columns
]
pedestal_candidates = sample_df.loc[
    sample_df["n_peaks"].eq(0), sampled_event_columns
]
peak_candidates = sample_df.loc[
    sample_df["signal_good"]
    & sample_df["snr"].ge(sample_peak_snr_min)
    & sample_df["peak_amplitude_mV"].le(sample_peak_amplitude_max_mV),
    sampled_event_columns,
]
signal_noise_like_candidates = sample_df.loc[
    sample_df["signal_noise_like"], sampled_event_columns
]
signal_off_time_candidates = sample_df.loc[
    sample_df["signal_off_time"], sampled_event_columns
]
signal_oscillatory_candidates = sample_df.loc[
    sample_df["signal_oscillatory"], sampled_event_columns
]
small_additional_peak_candidates = sample_df.loc[
    sample_df["signal_with_small_additional_peaks"], sampled_event_columns
]
small_additional_peak_blue_candidates = sample_df.loc[
    sample_df["signal_with_small_additional_peaks"]
    & sample_df["signal_good"]
    & sample_df["snr"].ge(sample_peak_snr_min)
    & sample_df["peak_amplitude_mV"].le(sample_peak_amplitude_max_mV),
    sampled_event_columns,
]
generic_bad_shape_candidates = sample_df.loc[
    sample_df["signal_bad_generic_shape"], sampled_event_columns
]
multi_peak_candidates = sample_df.loc[
    sample_df["n_peaks"].gt(1), sampled_event_columns
]
wide_peak_candidates = peak_candidates.loc[
    peak_candidates["peak_width_ns"].gt(wide_peak_diagnostic_min_width_ns)
]
n_multi_peak_events = len(multi_peak_candidates)
n_oscillatory_events = len(signal_oscillatory_candidates)
n_small_additional_peak_events = len(small_additional_peak_candidates)
n_small_additional_peak_blue_events = len(small_additional_peak_blue_candidates)
n_generic_bad_shape_events = len(generic_bad_shape_candidates)
n_wide_peak_events = len(wide_peak_candidates)
signal_like_bad_shape_count = (
    int(sample_df["signal_like_bad_shape"].sum())
    if "signal_like_bad_shape" in sample_df.columns else None
)
n_cached_signal_good_events = int(sample_df["signal_good"].sum())
n_pedestal_candidate_events = len(pedestal_candidates)
n_peak_candidate_events = len(peak_candidates)
n_signal_noise_like_events = len(signal_noise_like_candidates)
n_signal_off_time_events = len(signal_off_time_candidates)
n_cached_events = len(sample_df)
n_multi_peak_waveforms = min(
    multi_peak_diagnostic_max_waveforms, n_multi_peak_events
)
n_oscillatory_waveforms = min(
    oscillatory_diagnostic_max_waveforms, n_oscillatory_events
)
n_small_additional_peak_waveforms = min(
    small_additional_peak_diagnostic_max_waveforms,
    n_small_additional_peak_events,
)
n_small_additional_peak_blue_waveforms = min(
    small_additional_peak_blue_diagnostic_max_waveforms,
    n_small_additional_peak_blue_events,
)
n_generic_bad_shape_waveforms = min(
    generic_bad_shape_diagnostic_max_waveforms, n_generic_bad_shape_events
)
if max_pedestal_waveforms_for_qdc < 0 or max_peak_waveforms_for_qdc < 0:
    raise ValueError("Maximum QDC waveform counts must be non-negative")
n_pedestal_waveforms = min(max_pedestal_waveforms_for_qdc, n_pedestal_candidate_events)
n_peak_waveforms = min(max_peak_waveforms_for_qdc, n_peak_candidate_events)
if n_pedestal_waveforms < max_pedestal_waveforms_for_qdc:
    print(
        f"Pedestal QDC sample capped at all {n_pedestal_waveforms:,} available events "
        f"(requested maximum {max_pedestal_waveforms_for_qdc:,})."
    )
if n_peak_waveforms < max_peak_waveforms_for_qdc:
    print(
        f"Peak QDC sample capped at all {n_peak_waveforms:,} available events "
        f"(requested maximum {max_peak_waveforms_for_qdc:,})."
    )
if n_pedestal_waveforms_to_plot < 0 or n_peak_waveforms_to_plot < 0:
    raise ValueError("Waveform plot counts must be non-negative")
if n_pedestal_waveforms_to_plot > n_pedestal_waveforms:
    print(
        f"Pedestal waveform overlay capped at {n_pedestal_waveforms:,} available "
        f"events (requested {n_pedestal_waveforms_to_plot:,})."
    )
    n_pedestal_waveforms_to_plot = n_pedestal_waveforms
if n_peak_waveforms_to_plot > n_peak_waveforms:
    print(
        f"Signal waveform overlay capped at {n_peak_waveforms:,} available "
        f"events (requested {n_peak_waveforms_to_plot:,})."
    )
    n_peak_waveforms_to_plot = n_peak_waveforms
sampled_pedestals = pedestal_candidates.sample(n=n_pedestal_waveforms, random_state=sample_seed, replace=False)
sampled_peaks = peak_candidates.sample(n=n_peak_waveforms, random_state=sample_seed + 1, replace=False)
sampled_multi_peaks = multi_peak_candidates.sample(
    n=n_multi_peak_waveforms, random_state=sample_seed + 2, replace=False
)
sampled_oscillatory = signal_oscillatory_candidates.sample(
    n=n_oscillatory_waveforms, random_state=sample_seed + 3, replace=False
)
sampled_small_additional_peaks = small_additional_peak_candidates.sample(
    n=n_small_additional_peak_waveforms,
    random_state=sample_seed + 4,
    replace=False,
)
sampled_small_additional_peak_blue = small_additional_peak_blue_candidates.sample(
    n=n_small_additional_peak_blue_waveforms,
    random_state=sample_seed + 6,
    replace=False,
)
sampled_generic_bad_shape = generic_bad_shape_candidates.sample(
    n=n_generic_bad_shape_waveforms, random_state=sample_seed + 5, replace=False
)
del (
    pedestal_candidates, peak_candidates, signal_noise_like_candidates,
    signal_off_time_candidates, signal_oscillatory_candidates,
    small_additional_peak_candidates,
    small_additional_peak_blue_candidates, generic_bad_shape_candidates,
    multi_peak_candidates, wide_peak_candidates,
)
gc.collect()
sampled_events = pd.concat(
    [
        sampled_pedestals, sampled_peaks, sampled_multi_peaks,
        sampled_oscillatory, sampled_small_additional_peaks,
        sampled_small_additional_peak_blue,
        sampled_generic_bad_shape,
    ],
    ignore_index=True,
)

# Resolve moved raw data by basename when an older cache contains stale paths.
source_by_name = resolve_cached_event_files(
    sample_manifest["source_files"], search_roots=(Path("PMT_Data"),)
)
qdc_timing_exposure = None
if qdc_use_timing_rate_units:
    qdc_timing_exposure = load_acquisition_timing_exposure(
        source_by_name.values(),
        acquisition=sample_acquisition,
        time_column=qdc_timing_time_column,
    )
if qdc_timing_exposure is None:
    print(
        "No matching timing CSV found; QDC histograms will use "
        "raw waveform counts."
    )
    qdc_live_time_s = None
else:
    qdc_live_time_s = qdc_timing_exposure["live_time_s"]
    print(
        f"QDC rate exposure: {qdc_live_time_s:.6g} s from "
        f"{qdc_timing_exposure['n_files']} files in "
        f"{qdc_timing_exposure['timing_csv_path']} "
        f"({qdc_timing_exposure['logged_segments']:,} logged events; "
        f"overall trigger rate "
        f"{qdc_timing_exposure['logged_rate_evt_s']:.6g} /s)."
    )
qdc_pedestal_sampling_scale = (
    n_pedestal_candidate_events / n_pedestal_waveforms
    if n_pedestal_waveforms else 0.0
)
qdc_signal_sampling_scale = (
    n_peak_candidate_events / n_peak_waveforms if n_peak_waveforms else 0.0
)
qdc_pedestal_event_weight_per_s = (
    None if qdc_live_time_s is None
    else qdc_pedestal_sampling_scale / qdc_live_time_s
)
qdc_signal_event_weight_per_s = (
    None if qdc_live_time_s is None
    else qdc_signal_sampling_scale / qdc_live_time_s
)
if qdc_live_time_s is not None and (
    qdc_pedestal_sampling_scale > 1.0 or qdc_signal_sampling_scale > 1.0
):
    print(
        f"Rate histograms correct for random QDC sampling: pedestal "
        f"scale={qdc_pedestal_sampling_scale:.6g}, signal "
        f"scale={qdc_signal_sampling_scale:.6g}."
    )
qdc_histogram_y_label = (
    "Density" if qdc_histogram_density
    else ("Waveforms / s / bin" if qdc_live_time_s is not None else "Waveforms")
)

def qdc_histogram_weights(values, population):
    if qdc_histogram_density or qdc_live_time_s is None:
        return None
    event_weight = {
        "pedestal": qdc_pedestal_event_weight_per_s,
        "signal": qdc_signal_event_weight_per_s,
    }[population]
    return np.full(len(values), event_weight, dtype=float)

def combined_qdc_histogram_weights(pedestal_values, signal_values):
    if qdc_histogram_density or qdc_live_time_s is None:
        return None
    return np.concatenate([
        qdc_histogram_weights(pedestal_values, "pedestal"),
        qdc_histogram_weights(signal_values, "signal"),
    ])
sampled_events["event_file"] = sampled_events["event_file"].map(
    lambda path: str(source_by_name[Path(path).name])
)
if n_noisiest_kept_waveforms_to_plot < 0:
    raise ValueError("n_noisiest_kept_waveforms_to_plot must be non-negative")
finite_rms_events = sample_df.loc[
    np.isfinite(sample_df["baseline_rms_mV"]), sampled_event_columns
].copy()
n_noisiest_kept_waveforms = min(
    n_noisiest_kept_waveforms_to_plot, len(finite_rms_events)
)
noisiest_kept_events = finite_rms_events.nlargest(
    n_noisiest_kept_waveforms, "baseline_rms_mV"
).copy()
noisiest_kept_events["event_file"] = noisiest_kept_events["event_file"].map(
    lambda path: str(source_by_name[Path(path).name])
)
del sample_df, finite_rms_events
gc.collect()

reference_time_ns = None
reference_mV = None
reference_path_value = sample_preprocessing["baseline_reference_path"]
if reference_path_value is not None:
    reference = load_baseline_reference(reference_path_value)
    reference_time_ns = reference["time_ns"]
    reference_mV = reference["baseline_template_mV"]

# Learn the signal-shape template from only a small targeted high-SNR subset.
# The subsequent full QDC stream can then calculate shape metrics inline,
# avoiding a second full read of every pedestal and signal waveform.
shape_template_candidates = sampled_peaks.loc[
    sampled_peaks["snr"].ge(shape_template_reference_snr)
].head(shape_template_reference_waveforms).copy()
shape_template_candidates["event_file"] = shape_template_candidates[
    "event_file"
].map(lambda path: str(source_by_name[Path(path).name]))
shape_template_relative_time_ns = None
signal_shape_template = None
n_shape_template_references = 0
if len(shape_template_candidates):
    shape_reference_time_ns, shape_reference_waveforms_mV = load_event_waveforms(
        shape_template_candidates,
        channel=sample_preprocessing["channel"],
        baseline_window_ns=sample_baseline_window_ns,
        baseline_reference_time_ns=reference_time_ns,
        baseline_reference_mV=reference_mV,
    )
    shape_dt_ns = float(np.mean(np.diff(shape_reference_time_ns)))
    shape_template_relative_time_ns = np.arange(
        shape_template_relative_window_ns[0],
        shape_template_relative_window_ns[1] + 0.5 * shape_dt_ns,
        shape_dt_ns,
    )
    shape_reference_shapes = []
    for (_, reference_event), reference_waveform_mV in zip(
        shape_template_candidates.iterrows(), shape_reference_waveforms_mV
    ):
        sample_times = (
            float(reference_event["peak_time_ns"])
            + shape_template_relative_time_ns
        )
        if (
            sample_times[0] < shape_reference_time_ns[0]
            or sample_times[-1] > shape_reference_time_ns[-1]
        ):
            continue
        aligned = np.interp(
            sample_times, shape_reference_time_ns, -reference_waveform_mV
        )
        amplitude = float(np.max(aligned))
        if amplitude > 0.0:
            shape_reference_shapes.append(aligned / amplitude)
    n_shape_template_references = len(shape_reference_shapes)
    if n_shape_template_references >= shape_template_min_reference_waveforms:
        signal_shape_template = np.median(
            np.asarray(shape_reference_shapes), axis=0
        )
    del shape_reference_waveforms_mV
    shape_reference_shapes.clear()
    gc.collect()
if signal_shape_template is None:
    print(
        f"Waveform-shape metrics disabled: only "
        f"{n_shape_template_references} usable high-SNR references; "
        f"need {shape_template_min_reference_waveforms}."
    )
else:
    print(
        f"Learned signal-shape template from "
        f"{n_shape_template_references} targeted high-SNR waveforms."
    )

# Plot only the requested number of highest-RMS waveforms that survived
# Selection's global RMS cut. Raw files are read only for these rows.
if n_noisiest_kept_waveforms:
    noisiest_time_ns, noisiest_waveforms_mV = load_event_waveforms(
        noisiest_kept_events,
        channel=sample_preprocessing["channel"],
        baseline_window_ns=sample_baseline_window_ns,
        baseline_reference_time_ns=reference_time_ns,
        baseline_reference_mV=reference_mV,
    )
    noisiest_ncols = 2
    noisiest_nrows = int(np.ceil(n_noisiest_kept_waveforms / noisiest_ncols))
    fig_noisiest, noisiest_axes = plt.subplots(
        noisiest_nrows, noisiest_ncols,
        figsize=(14, 4.2 * noisiest_nrows), squeeze=False, sharex=True,
    )
    rms_cut_threshold_value = (
        None if sample_baseline_rms_cut is None
        else sample_baseline_rms_cut.get("threshold_mV")
    )
    rms_cut_threshold_mV = (
        np.nan if rms_cut_threshold_value is None else float(rms_cut_threshold_value)
    )
    for ax, (_, event), waveform_mV in zip(
        noisiest_axes.ravel(), noisiest_kept_events.iterrows(), noisiest_waveforms_mV
    ):
        event_rms_mV = float(event["baseline_rms_mV"])
        n_event_peaks = int(event["n_peaks"])
        if bool(event.get("signal_oscillatory", False)):
            event_class = "oscillatory: multiple qualifying candidates"
        elif bool(event.get("signal_bad_generic_shape", False)):
            event_class = "isolated peak outside generic shape bounds"
        elif bool(event.get("signal_with_small_additional_peaks", False)):
            event_class = "dominant signal with small additional candidates"
        elif bool(event.get("signal_like_bad_shape", False)):
            event_class = "signal-like bad shape"
        elif bool(event.get("signal_noise_like", False)):
            event_class = "single peak, noise-like shape"
        elif bool(event.get("signal_off_time", False)):
            event_class = "single peak, good shape but off-time"
        elif bool(event.get("signal_good", False)):
            event_class = "good shape + trigger-aligned signal"
        elif n_event_peaks == 0:
            event_class = "pedestal-like"
        elif n_event_peaks == 1:
            event_class = "single peak"
        else:
            event_class = f"multi-peak ({n_event_peaks})"
        ax.plot(noisiest_time_ns, waveform_mV, color="tab:brown", linewidth=0.9)
        ax.axhline(0.0, color="black", linewidth=0.8)
        ax.axhline(
            event_rms_mV, color="tab:red", linestyle="--", alpha=0.8,
            label="±1 baseline RMS",
        )
        ax.axhline(-event_rms_mV, color="tab:red", linestyle="--", alpha=0.8)
        ax.axvspan(
            *sample_baseline_window_ns, color="tab:orange", alpha=0.12,
            label="baseline RMS window",
        )
        threshold_text = (
            "" if not np.isfinite(rms_cut_threshold_mV)
            else f"; {100 * event_rms_mV / rms_cut_threshold_mV:.2f}% of cutoff"
        )
        ax.set_title(
            f"{event_class}; RMS={event_rms_mV:.6g} mV{threshold_text}\n"
            f"segment {int(event['event_segment'])}", fontsize=9,
        )
        ax.set(xlabel="Time [ns]", ylabel="Voltage [mV]")
        ax.grid(True, which="both", alpha=0.35)
    for ax in noisiest_axes.ravel()[n_noisiest_kept_waveforms:]:
        ax.set_visible(False)
    handles, labels = noisiest_axes.ravel()[0].get_legend_handles_labels()
    fig_noisiest.legend(handles, labels, loc="lower center", fontsize=8)
    fig_noisiest.suptitle(
        f"Noisiest {n_noisiest_kept_waveforms} waveforms remaining after "
        f"the baseline-RMS cut — baseline region highlighted"
    )
    fig_noisiest.tight_layout(rect=(0, 0.04, 1, 0.96))
    plt.show()

pedestal_qdc_window_ns = (0.0, 80.0)
sampled_events["qdc_population"] = (
    ["pedestal"] * n_pedestal_waveforms
    + ["peak"] * n_peak_waveforms
    + ["multi_peak"] * n_multi_peak_waveforms
    + ["oscillatory"] * n_oscillatory_waveforms
    + ["small_additional_peak"] * n_small_additional_peak_waveforms
    + ["small_additional_peak_blue"] * n_small_additional_peak_blue_waveforms
    + ["generic_bad_shape"] * n_generic_bad_shape_waveforms
)
streamed_qdc = stream_event_qdc_diagnostics(
    sampled_events,
    channel=sample_preprocessing["channel"],
    baseline_window_ns=sample_baseline_window_ns,
    baseline_reference_time_ns=reference_time_ns,
    baseline_reference_mV=reference_mV,
    peak_snr_threshold=sample_preprocessing["peak_snr_threshold"],
    peak_prominence_snr=sample_preprocessing["peak_prominence_snr"],
    peak_distance_samples=sample_preprocessing["peak_distance_samples"],
    peak_width_samples=sample_preprocessing["peak_width_samples"],
    pedestal_qdc_window_ns=pedestal_qdc_window_ns,
    pedestal_full_waveform_baseline_sidebands_ns=(
        pedestal_full_waveform_baseline_sidebands_ns
    ),
    pedestal_waveforms_to_keep=n_pedestal_waveforms_to_plot,
    peak_waveforms_to_keep=n_peak_waveforms_to_plot,
    multi_peak_waveforms_to_keep=n_multi_peak_waveforms,
    oscillatory_waveforms_to_keep=n_oscillatory_waveforms,
    small_additional_peak_waveforms_to_keep=(
        n_small_additional_peak_waveforms
    ),
    small_additional_peak_blue_waveforms_to_keep=(
        n_small_additional_peak_blue_waveforms
    ),
    signal_template_reference_snr=shape_template_reference_snr,
    signal_template_reference_waveforms_to_keep=0,
    signal_shape_template_relative_time_ns=shape_template_relative_time_ns,
    signal_shape_template_normalized=signal_shape_template,
    shape_pedestal_time_blocks=shape_pedestal_time_blocks,
    shape_pedestal_edge_blocks=shape_pedestal_edge_blocks,
    shape_pedestal_local_rms_window_ns=(
        shape_pedestal_local_rms_window_ns
    ),
    shape_pedestal_rms_variation_max=(
        shape_pedestal_rms_variation_max
    ),
    shape_worst_waveforms_to_keep=shape_worst_waveforms_to_plot,
    shape_signal_reject_upper_fraction=shape_signal_reject_upper_fraction,
    shape_pedestal_growth_ratio_max=shape_pedestal_growth_ratio_max,
    shape_boundary_waveforms_to_keep=shape_worst_waveforms_to_plot,
    generic_bad_shape_waveforms_to_keep=n_generic_bad_shape_waveforms,
    pedestal_diagnostic_qdc_range=pedestal_qdc_diagnostic_range_mV_ns,
    pedestal_diagnostic_max_waveforms=pedestal_qdc_diagnostic_max_waveforms,
    peak_diagnostic_qdc_range=peak_low_qdc_diagnostic_range_mV_ns,
    peak_diagnostic_max_waveforms=peak_low_qdc_diagnostic_max_waveforms,
    wide_peak_diagnostic_min_width_ns=wide_peak_diagnostic_min_width_ns,
    wide_peak_diagnostic_max_waveforms=wide_peak_diagnostic_max_waveforms,
    read_chunk_size=qdc_read_chunk_size,
)
sample_time_ns = streamed_qdc["time_ns"]
pedestal_plot_records = streamed_qdc["plot_records"]["pedestal"]
peak_plot_records = streamed_qdc["plot_records"]["peak"]
multi_peak_plot_records = streamed_qdc["plot_records"]["multi_peak"]
oscillatory_plot_records = streamed_qdc["plot_records"]["oscillatory"]
small_additional_peak_plot_records = streamed_qdc["plot_records"][
    "small_additional_peak"
]
small_additional_peak_blue_plot_records = streamed_qdc["plot_records"][
    "small_additional_peak_blue"
]
generic_bad_shape_plot_records = streamed_qdc["plot_records"]["generic_bad_shape"]
pedestal_diagnostic_records = streamed_qdc["pedestal_diagnostic_records"]
peak_low_qdc_diagnostic_records = streamed_qdc["peak_diagnostic_records"]
wide_peak_diagnostic_records = streamed_qdc["wide_peak_diagnostic_records"]
shape_quality_results = (
    None
    if signal_shape_template is None
    else {
        "signal_template_mismatch": streamed_qdc[
            "signal_template_mismatch"
        ],
        "pedestal_noise_growth_ratio": streamed_qdc[
            "pedestal_noise_growth_ratio"
        ],
        "pedestal_local_rms_relative_span": streamed_qdc[
            "pedestal_local_rms_relative_span"
        ],
        "worst_signal_records": streamed_qdc[
            "worst_signal_shape_records"
        ],
        "worst_pedestal_records": streamed_qdc[
            "worst_pedestal_growth_records"
        ],
        "worst_pedestal_rms_variation_records": streamed_qdc[
            "worst_pedestal_rms_variation_records"
        ],
        "signal_boundary_candidates": streamed_qdc[
            "signal_shape_boundary_candidates"
        ],
        "accepted_pedestal_records": streamed_qdc[
            "accepted_pedestal_growth_records"
        ],
        "accepted_pedestal_rms_variation_records": streamed_qdc[
            "accepted_pedestal_rms_variation_records"
        ],
    }
)
pedestal_qdc_values = streamed_qdc["pedestal_qdc_mV_ns"]
pedestal_full_waveform_qdc_values = streamed_qdc[
    "pedestal_full_waveform_qdc_mV_ns"
]
peak_qdc_values = streamed_qdc["peak_qdc_mV_ns"]
pedestal_raw_full_waveform_qdc_values = streamed_qdc[
    "pedestal_raw_full_waveform_qdc_mV_ns"
]
peak_raw_full_waveform_qdc_values = streamed_qdc[
    "peak_raw_full_waveform_qdc_mV_ns"
]
pedestal_max_amplitudes_mV = streamed_qdc["pedestal_max_amplitude_mV"]
peak_max_amplitudes_mV = streamed_qdc["peak_max_amplitude_mV"]

# Apply the normalized-template mismatch cut to the final blue Testing
# population. Preserve the pre-cut arrays for the diagnostics PDF.
shape_cache_signal_events_before_mismatch_cut = sampled_peaks.loc[:, [
    "event_file", "event_segment", "peak_time_ns",
    "peak_width_ns", sample_fixed_charge_column,
]].copy()
peak_qdc_values_before_mismatch_cut = np.asarray(
    peak_qdc_values, dtype=float
).copy()
peak_raw_full_waveform_qdc_values_before_mismatch_cut = np.asarray(
    peak_raw_full_waveform_qdc_values, dtype=float
).copy()
peak_max_amplitudes_mV_before_mismatch_cut = np.asarray(
    peak_max_amplitudes_mV, dtype=float
).copy()
n_peak_waveforms_before_mismatch_cut = len(sampled_peaks)
if not 0.0 <= shape_signal_reject_upper_fraction < 1.0:
    raise ValueError(
        "shape_signal_reject_upper_fraction must be in [0, 1)"
    )
if shape_quality_results is None:
    signal_template_mismatch_for_cut = np.full(
        n_peak_waveforms_before_mismatch_cut, np.nan
    )
    signal_mismatch_threshold = np.nan
    signal_shape_keep = np.ones(
        n_peak_waveforms_before_mismatch_cut, dtype=bool
    )
    signal_shape_cut_applied = False
else:
    signal_template_mismatch_for_cut = np.asarray(
        shape_quality_results["signal_template_mismatch"], dtype=float
    )
    if len(signal_template_mismatch_for_cut) != n_peak_waveforms_before_mismatch_cut:
        raise ValueError(
            "Signal mismatch/event alignment changed: "
            f"{len(signal_template_mismatch_for_cut)} metrics versus "
            f"{n_peak_waveforms_before_mismatch_cut} signal events"
        )
    finite_signal_mismatch_for_cut = np.isfinite(
        signal_template_mismatch_for_cut
    )
    if np.any(finite_signal_mismatch_for_cut):
        signal_mismatch_threshold = float(np.quantile(
            signal_template_mismatch_for_cut[finite_signal_mismatch_for_cut],
            1.0 - shape_signal_reject_upper_fraction,
        ))
        signal_shape_keep = (
            finite_signal_mismatch_for_cut
            & (
                signal_template_mismatch_for_cut
                <= signal_mismatch_threshold
            )
        )
        signal_shape_cut_applied = True
    else:
        signal_mismatch_threshold = np.nan
        signal_shape_keep = np.ones(
            n_peak_waveforms_before_mismatch_cut, dtype=bool
        )
        signal_shape_cut_applied = False
n_signal_after_mismatch_cut = int(np.sum(signal_shape_keep))
signal_mismatch_survival_percent = (
    100.0 * n_signal_after_mismatch_cut
    / n_peak_waveforms_before_mismatch_cut
    if n_peak_waveforms_before_mismatch_cut else np.nan
)
if signal_shape_cut_applied:
    signal_shape_survival_annotation = (
        f"mismatch-cut survival: {n_signal_after_mismatch_cut:,}/"
        f"{n_peak_waveforms_before_mismatch_cut:,} "
        f"({signal_mismatch_survival_percent:.2f}%)"
    )
    print(
        f"Applied signal mismatch cut at <= {signal_mismatch_threshold:.6g}: "
        f"kept {n_signal_after_mismatch_cut:,}/"
        f"{n_peak_waveforms_before_mismatch_cut:,} signal events "
        f"({signal_mismatch_survival_percent:.2f}% survival). "
        "Pedestal growth and local-RMS cuts were not applied."
    )
else:
    signal_shape_survival_annotation = "signal mismatch cut unavailable"
    print(
        "Signal mismatch cut was not applied because no finite template "
        "mismatch metrics were available. Pedestal shape cuts were also not applied."
    )

sampled_peaks = sampled_peaks.loc[
    signal_shape_keep
].copy()
peak_qdc_values = peak_qdc_values_before_mismatch_cut[signal_shape_keep]
peak_raw_full_waveform_qdc_values = (
    peak_raw_full_waveform_qdc_values_before_mismatch_cut[signal_shape_keep]
)
peak_max_amplitudes_mV = (
    peak_max_amplitudes_mV_before_mismatch_cut[signal_shape_keep]
)
n_peak_waveforms = len(sampled_peaks)

# Remove rejected signal rows from later overlap/tail sampling while leaving
# every pedestal and every non-blue diagnostic population untouched.
sampled_peak_positions = np.flatnonzero(
    sampled_events["qdc_population"].eq("peak").to_numpy()
)
sampled_pedestal_positions = np.flatnonzero(
    sampled_events["qdc_population"].eq("pedestal").to_numpy()
)
if len(sampled_peak_positions) != len(signal_shape_keep):
    raise ValueError(
        "Sampled signal/QDC population alignment changed before mismatch cut"
    )
sampled_peak_positions_after_mismatch_cut = sampled_peak_positions[
    signal_shape_keep
]

def signal_record_passes_mismatch(record):
    population_order = int(record["population_order"])
    return (
        0 <= population_order < len(signal_shape_keep)
        and bool(signal_shape_keep[population_order])
    )

peak_plot_records = [
    record for record in peak_plot_records
    if signal_record_passes_mismatch(record)
][:n_peak_waveforms_to_plot]
peak_low_qdc_diagnostic_records = [
    record for record in peak_low_qdc_diagnostic_records
    if signal_record_passes_mismatch(record)
]
wide_peak_diagnostic_records = [
    record for record in wide_peak_diagnostic_records
    if signal_record_passes_mismatch(record)
]

# Save the current voltage's reduced three-method arrays while they are already
# in memory. The optional comparison cell then only decides which caches to plot.
current_qdc_method_cache_version = 5
current_qdc_method_fixed_column = sample_fixed_charge_column
current_qdc_method_cache_dir = (
    shape_diagnostics_pdf_path.parent / "voltage_qdc_method_cache"
)
current_qdc_method_cache_path = (
    current_qdc_method_cache_dir
    / f"{sample_acquisition}_qdc_methods.npz"
)
current_qdc_method_source_mtime_ns = sample_cache_path.stat().st_mtime_ns
current_qdc_method_required_keys = {
    "fixed", "event_specific", "raw_full_trace",
    "fixed_pedestal", "fixed_signal_good",
    "event_specific_pedestal", "event_specific_signal_good",
    "raw_full_trace_pedestal", "raw_full_trace_signal_good",
    "live_time_s", "pedestal_event_weight_per_s",
    "signal_good_event_weight_per_s",
    "signal_mismatch_reject_upper_fraction",
    "signal_mismatch_threshold", "signal_mismatch_survival_percent",
}
current_qdc_method_cache_valid = False
if current_qdc_method_cache_path.is_file():
    try:
        with np.load(current_qdc_method_cache_path, allow_pickle=False) as saved:
            current_qdc_method_cache_valid = (
                int(saved["cache_version"].item())
                == current_qdc_method_cache_version
                and int(saved["source_cache_mtime_ns"].item())
                == current_qdc_method_source_mtime_ns
                and current_qdc_method_required_keys.issubset(saved.files)
                and np.isclose(
                    float(saved["signal_mismatch_reject_upper_fraction"].item()),
                    shape_signal_reject_upper_fraction,
                )
                and np.isclose(
                    float(saved["signal_mismatch_threshold"].item()),
                    signal_mismatch_threshold, equal_nan=True,
                )
            )
    except (OSError, KeyError, TypeError, ValueError):
        current_qdc_method_cache_valid = False
expected_qdc_method_pedestals = n_pedestal_candidate_events
expected_qdc_method_signals = n_peak_candidate_events
current_qdc_method_sample_complete = (
    n_pedestal_waveforms == expected_qdc_method_pedestals
    and n_peak_waveforms_before_mismatch_cut == expected_qdc_method_signals
)
if current_qdc_method_cache_valid:
    print(f"Current-voltage QDC method cache is current: {current_qdc_method_cache_path}")
elif not sample_fixed_charge_available:
    print(
        f"Skipping current-voltage QDC method cache: missing "
        f"{current_qdc_method_fixed_column!r}."
    )
elif not current_qdc_method_sample_complete:
    print(
        "Skipping current-voltage QDC method cache because the main QDC "
        f"sample is incomplete: pedestals {n_pedestal_waveforms:,}/"
        f"{expected_qdc_method_pedestals:,}, signal_good "
        f"{n_peak_waveforms_before_mismatch_cut:,}/"
        f"{expected_qdc_method_signals:,} before the mismatch cut. "
        "Raise the QDC waveform caps or remove signal sampling filters."
    )
else:
    current_qdc_method_populations = {
        "fixed": {
            "pedestal": sampled_pedestals[
                current_qdc_method_fixed_column
            ].to_numpy(dtype=float),
            "signal_good": sampled_peaks[
                current_qdc_method_fixed_column
            ].to_numpy(dtype=float),
        },
        "event_specific": {
            "pedestal": np.asarray(pedestal_qdc_values, dtype=float),
            "signal_good": np.asarray(peak_qdc_values, dtype=float),
        },
        "raw_full_trace": {
            "pedestal": np.asarray(
                pedestal_raw_full_waveform_qdc_values, dtype=float
            ),
            "signal_good": np.asarray(
                peak_raw_full_waveform_qdc_values, dtype=float
            ),
        },
    }
    current_qdc_method_populations = {
        method: {
            population: values[np.isfinite(values)]
            for population, values in populations.items()
        }
        for method, populations in current_qdc_method_populations.items()
    }
    current_qdc_method_combined = {
        method: np.concatenate(list(populations.values()))
        for method, populations in current_qdc_method_populations.items()
    }
    current_qdc_method_cache_dir.mkdir(parents=True, exist_ok=True)
    current_qdc_method_temporary_path = (
        current_qdc_method_cache_path.with_suffix(
            current_qdc_method_cache_path.suffix + ".partial"
        )
    )
    with current_qdc_method_temporary_path.open("wb") as reduced_file:
        np.savez_compressed(
            reduced_file,
            cache_version=np.asarray(current_qdc_method_cache_version),
            source_cache_mtime_ns=np.asarray(
                current_qdc_method_source_mtime_ns
            ),
            live_time_s=np.asarray(
                np.nan if qdc_live_time_s is None else qdc_live_time_s
            ),
            pedestal_event_weight_per_s=np.asarray(
                np.nan if qdc_pedestal_event_weight_per_s is None
                else qdc_pedestal_event_weight_per_s
            ),
            signal_good_event_weight_per_s=np.asarray(
                np.nan if qdc_signal_event_weight_per_s is None
                else qdc_signal_event_weight_per_s
            ),
            signal_mismatch_reject_upper_fraction=np.asarray(
                shape_signal_reject_upper_fraction
            ),
            signal_mismatch_threshold=np.asarray(
                signal_mismatch_threshold
            ),
            signal_mismatch_survival_percent=np.asarray(
                signal_mismatch_survival_percent
            ),
            **current_qdc_method_combined,
            **{
                f"{method}_{population}": values
                for method, populations in (
                    current_qdc_method_populations.items()
                )
                for population, values in populations.items()
            },
        )
    current_qdc_method_temporary_path.replace(
        current_qdc_method_cache_path
    )
    print(
        f"Saved current-voltage QDC method cache to "
        f"{current_qdc_method_cache_path}"
    )

# Keep every finite pedestal QDC value in all separate and combined plots.
# No charge-tail percentile is applied.
pedestal_qdc_plot_values = np.asarray(pedestal_qdc_values, dtype=float)
pedestal_qdc_plot_values = pedestal_qdc_plot_values[
    np.isfinite(pedestal_qdc_plot_values)
]
pedestal_qdc_plot_total = len(pedestal_qdc_plot_values)
pedestal_full_qdc_plot_values = np.asarray(
    pedestal_full_waveform_qdc_values, dtype=float
)
pedestal_full_qdc_plot_values = pedestal_full_qdc_plot_values[
    np.isfinite(pedestal_full_qdc_plot_values)
]
pedestal_full_qdc_plot_total = len(pedestal_full_qdc_plot_values)
if not pedestal_qdc_plot_total:
    print(
        "No pedestal-classified events are available; pedestal-only diagnostics "
        "will be empty and signal diagnostics will continue."
    )
combined_qdc_values = np.concatenate([pedestal_qdc_plot_values, peak_qdc_values])

# Identify the central shared probability mass of the two QDC populations.
def qdc_histogram_overlap_interval(pedestal_values, signal_values):
    pedestal_values = np.asarray(pedestal_values, dtype=float)
    signal_values = np.asarray(signal_values, dtype=float)
    pedestal_values = pedestal_values[np.isfinite(pedestal_values)]
    signal_values = signal_values[np.isfinite(signal_values)]
    if not len(pedestal_values) or not len(signal_values):
        return None
    display_low, display_high = map(float, qdc_overlap_display_quantiles)
    mass_low, mass_high = map(float, qdc_overlap_mass_quantiles)
    if not 0.0 <= display_low < display_high <= 1.0:
        raise ValueError("qdc_overlap_display_quantiles must satisfy 0 <= low < high <= 1")
    if not 0.0 <= mass_low < mass_high <= 1.0:
        raise ValueError("qdc_overlap_mass_quantiles must satisfy 0 <= low < high <= 1")
    combined_finite = np.concatenate([pedestal_values, signal_values])
    range_low, range_high = np.quantile(
        combined_finite, (display_low, display_high)
    )
    if not np.isfinite(range_low) or not np.isfinite(range_high):
        return None
    if range_low == range_high:
        padding = max(1e-6, abs(float(range_low)) * 0.01)
        range_low, range_high = range_low - padding, range_high + padding
    edges = np.linspace(
        float(range_low), float(range_high), int(qdc_overlap_histogram_bins) + 1
    )
    pedestal_counts, _ = np.histogram(pedestal_values, bins=edges)
    signal_counts, _ = np.histogram(signal_values, bins=edges)
    if not pedestal_counts.sum() or not signal_counts.sum():
        return None
    pedestal_probability = pedestal_counts / pedestal_counts.sum()
    signal_probability = signal_counts / signal_counts.sum()
    shared_probability = np.minimum(pedestal_probability, signal_probability)
    overlap_coefficient = float(shared_probability.sum())
    if overlap_coefficient <= 0.0:
        return None
    shared_cdf = np.cumsum(shared_probability) / overlap_coefficient
    first_bin = min(
        int(np.searchsorted(shared_cdf, mass_low, side="left")),
        len(shared_probability) - 1,
    )
    last_bin = min(
        int(np.searchsorted(shared_cdf, mass_high, side="left")),
        len(shared_probability) - 1,
    )
    return {
        "interval_mV_ns": (float(edges[first_bin]), float(edges[last_bin + 1])),
        "overlap_coefficient": overlap_coefficient,
        "edges": edges,
        "pedestal_probability": pedestal_probability,
        "signal_probability": signal_probability,
        "shared_probability": shared_probability,
    }

qdc_overlap = qdc_histogram_overlap_interval(
    pedestal_qdc_values, peak_qdc_values
)
if qdc_overlap is None:
    print("No finite histogram overlap was found between pedestal and signal QDC.")
else:
    overlap_low, overlap_high = qdc_overlap["interval_mV_ns"]
    signal_overlap_extension_fraction = float(
        qdc_overlap_signal_upper_extension_fraction
    )
    if signal_overlap_extension_fraction < 0.0:
        raise ValueError(
            "qdc_overlap_signal_upper_extension_fraction must be non-negative"
        )
    signal_diagnostic_high = overlap_high + signal_overlap_extension_fraction * (
        overlap_high - overlap_low
    )
    print(
        f"Automatic pedestal/signal QDC overlap region: "
        f"[{overlap_low:.6g}, {overlap_high:.6g}] mV ns; "
        f"histogram overlap coefficient={qdc_overlap['overlap_coefficient']:.4f}"
    )

    fig_overlap, ax_overlap = plt.subplots(figsize=(12, 6.5))
    overlap_edges = qdc_overlap["edges"]
    ax_overlap.stairs(
        qdc_overlap["pedestal_probability"], overlap_edges,
        color="tab:orange", linewidth=1.5, label="pedestal-like",
    )
    ax_overlap.stairs(
        qdc_overlap["signal_probability"], overlap_edges,
        color="tab:blue", linewidth=1.5, label="signal",
    )
    ax_overlap.stairs(
        qdc_overlap["shared_probability"], overlap_edges, fill=True,
        color="tab:purple", alpha=0.20, label="shared probability/bin",
    )
    ax_overlap.axvspan(
        overlap_low, overlap_high, color="tab:red", alpha=0.10,
        label=(
            f"central {100 * (qdc_overlap_mass_quantiles[1] - qdc_overlap_mass_quantiles[0]):g}% "
            "of shared mass"
        ),
    )
    if signal_diagnostic_high > overlap_high:
        ax_overlap.axvspan(
            overlap_high, signal_diagnostic_high, color="tab:blue", alpha=0.08,
            label="additional signal diagnostic range",
        )
    ax_overlap.set(
        xlabel="QDC [mV ns]", ylabel="Probability per bin",
        title=(
            f"{sample_acquisition}: automatic pedestal/signal QDC overlap\n"
            f"overlap coefficient={qdc_overlap['overlap_coefficient']:.4f}; "
            f"diagnostic interval=[{overlap_low:.4g}, {overlap_high:.4g}] mV ns"
        ),
    )
    ax_overlap.grid(True, alpha=0.3)
    ax_overlap.legend()
    fig_overlap.tight_layout()
    plt.show()

    def sample_qdc_overlap_events(
        events, event_positions, qdc_values, population, seed,
        selection_low, selection_high,
    ):
        event_positions = np.asarray(event_positions, dtype=int)
        qdc_values = np.asarray(qdc_values, dtype=float)
        if len(event_positions) != len(qdc_values):
            raise ValueError(
                f"{population} event/QDC alignment changed: "
                f"{len(event_positions)} events versus "
                f"{len(qdc_values)} charges"
            )
        in_overlap = (
            np.isfinite(qdc_values)
            & (qdc_values >= selection_low)
            & (qdc_values <= selection_high)
        )
        candidate_local_positions = np.flatnonzero(in_overlap)
        n_keep = min(
            int(qdc_overlap_diagnostic_max_waveforms_per_population),
            len(candidate_local_positions),
        )
        if not n_keep:
            return events.iloc[0:0].copy()
        rng = np.random.default_rng(seed)
        selected_local_positions = rng.choice(
            candidate_local_positions, size=n_keep, replace=False
        )
        selected = events.iloc[
            event_positions[selected_local_positions]
        ].copy()
        selected["overlap_qdc_mV_ns"] = qdc_values[
            selected_local_positions
        ]
        selected["overlap_population"] = population
        return selected
    overlap_pedestal_events = sample_qdc_overlap_events(
        sampled_events, sampled_pedestal_positions, pedestal_qdc_values,
        "pedestal",
        sample_seed + 101, overlap_low, overlap_high,
    )
    overlap_peak_events = sample_qdc_overlap_events(
        sampled_events, sampled_peak_positions_after_mismatch_cut,
        peak_qdc_values, "signal", sample_seed + 102,
        overlap_low, signal_diagnostic_high,
    )
    overlap_diagnostic_events = pd.concat(
        [overlap_pedestal_events, overlap_peak_events], ignore_index=True
    )
    print(
        f"QDC-overlap candidates: {len(overlap_pedestal_events):,} pedestal and "
        f"{len(overlap_peak_events):,} signal waveforms selected for plotting; "
        f"signal range extends to {signal_diagnostic_high:.6g} mV ns."
    )
    if len(overlap_diagnostic_events):
        overlap_time_ns, overlap_diagnostic_waveforms_mV = load_event_waveforms(
            overlap_diagnostic_events,
            channel=sample_preprocessing["channel"],
            baseline_window_ns=sample_baseline_window_ns,
            baseline_reference_time_ns=reference_time_ns,
            baseline_reference_mV=reference_mV,
        )
        overlap_records = []
        for (_, event), waveform_mV in zip(
            overlap_diagnostic_events.iterrows(),
            overlap_diagnostic_waveforms_mV,
        ):
            waveform_mV = np.asarray(waveform_mV, dtype=float)
            if event["overlap_population"] == "pedestal":
                waveform_mV = waveform_mV - np.mean(waveform_mV)
            overlap_records.append((event, waveform_mV))

        n_overlap_rows = max(
            len(overlap_pedestal_events), len(overlap_peak_events)
        )
        fig_overlap_waveforms, overlap_axes = plt.subplots(
            n_overlap_rows, 2, figsize=(15, 4.2 * n_overlap_rows),
            squeeze=False, sharex=True,
        )
        population_rows = {"pedestal": 0, "signal": 0}
        population_columns = {"pedestal": 0, "signal": 1}
        population_colors = {"pedestal": "tab:orange", "signal": "tab:blue"}
        for event, waveform_mV in overlap_records:
            population = event["overlap_population"]
            row = population_rows[population]
            population_rows[population] += 1
            ax = overlap_axes[row, population_columns[population]]
            ax.plot(
                overlap_time_ns, waveform_mV,
                color=population_colors[population], linewidth=1.0,
            )
            ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.6)
            ax.axvspan(
                *sample_baseline_window_ns, color="0.5", alpha=0.08,
                label="baseline window",
            )
            baseline_rms = float(event.get("baseline_rms_mV", np.nan))
            if np.isfinite(baseline_rms):
                ax.axhline(
                    -sample_preprocessing["peak_snr_threshold"] * baseline_rms,
                    color="tab:red", linestyle="--", linewidth=0.9,
                    alpha=0.7, label="peak-height threshold",
                )
            if population == "pedestal":
                ax.axvspan(
                    *pedestal_qdc_window_ns, color="tab:orange", alpha=0.07,
                    label="pedestal QDC window",
                )
            else:
                peak_time = float(event.get("peak_time_ns", np.nan))
                peak_width = float(event.get("peak_width_ns", np.nan))
                if np.isfinite(peak_time):
                    ax.axvline(
                        peak_time, color="tab:red", linestyle="--",
                        linewidth=1.1, label="accepted peak",
                    )
                if np.isfinite(peak_time) and np.isfinite(peak_width):
                    ax.axvspan(
                        peak_time - peak_width, peak_time + 2.0 * peak_width,
                        color="tab:blue", alpha=0.07,
                        label="signal QDC window",
                    )
            detail_lines = [
                f"QDC={float(event['overlap_qdc_mV_ns']):.4g} mV ns"
            ]
            amplitude = float(event.get("peak_amplitude_mV", np.nan))
            if np.isfinite(amplitude):
                detail_lines.append(f"amplitude={amplitude:.4g} mV")
            snr = float(event.get("snr", np.nan))
            if np.isfinite(snr):
                detail_lines.append(f"SNR={snr:.3g}")
            fwhm = float(event.get("peak_width_ns", np.nan))
            if np.isfinite(fwhm):
                detail_lines.append(f"FWHM={fwhm:.3g} ns")
            details = "; ".join(detail_lines)
            additional_fraction = float(event.get(
                "largest_additional_signal_like_peak_fraction", np.nan
            ))
            if np.isfinite(additional_fraction) and additional_fraction > 0:
                details += f"; additional/main={additional_fraction:.3g}"
            ax.text(
                0.98, 0.04, details, transform=ax.transAxes, ha="right",
                va="bottom", fontsize=8,
                bbox={"facecolor": "white", "alpha": 0.78, "edgecolor": "0.7"},
            )
            ax.set_title(
                f"{population}: {Path(event['event_file']).name}, "
                f"segment {int(event['event_segment'])}", fontsize=9,
            )
            ax.set(xlabel="Time [ns]", ylabel="Voltage [mV]")
            ax.grid(True, which="both", alpha=0.35)
        for population, column in population_columns.items():
            for row in range(population_rows[population], n_overlap_rows):
                overlap_axes[row, column].set_visible(False)
        visible_axes = [ax for ax in overlap_axes.ravel() if ax.get_visible()]
        legend_entries = {}
        for visible_ax in visible_axes:
            handles, labels = visible_ax.get_legend_handles_labels()
            legend_entries.update(
                (label, handle) for handle, label in zip(handles, labels) if label
            )
        fig_overlap_waveforms.legend(
            legend_entries.values(), legend_entries.keys(),
            loc="lower center", ncols=4, fontsize=8,
        )
        fig_overlap_waveforms.suptitle(
            f"{sample_acquisition}: QDC-overlap diagnostics — pedestal "
            f"[{overlap_low:.4g}, {overlap_high:.4g}], signal "
            f"[{overlap_low:.4g}, {signal_diagnostic_high:.4g}] mV ns"
        )
        fig_overlap_waveforms.tight_layout(rect=(0, 0.035, 1, 0.97))
        plt.show()

# Plot representative waveforms from each population's highest-QDC tail.
def sample_high_qdc_tail(
    events, event_positions, qdc_values, population, seed
):
    event_positions = np.asarray(event_positions, dtype=int)
    qdc_values = np.asarray(qdc_values, dtype=float)
    if len(event_positions) != len(qdc_values):
        raise ValueError(
            f"{population} event/QDC alignment changed: "
            f"{len(event_positions)} events versus {len(qdc_values)} charges"
        )
    finite = np.isfinite(qdc_values)
    n_finite = int(finite.sum())
    if not n_finite:
        return events.iloc[0:0].copy(), np.nan, 0.0, 0
    requested_fraction = float(qdc_high_tail_fraction)
    if not 0.0 < requested_fraction <= 1.0:
        raise ValueError("qdc_high_tail_fraction must be in (0, 1]")
    minimum_candidates = max(
        1, int(qdc_high_tail_min_candidates_per_population)
    )
    effective_fraction = min(
        1.0, max(requested_fraction, minimum_candidates / n_finite)
    )
    threshold = float(np.quantile(qdc_values[finite], 1.0 - effective_fraction))
    in_tail = finite & (qdc_values >= threshold)
    candidate_local_positions = np.flatnonzero(in_tail)
    n_plot = min(
        int(qdc_high_tail_waveforms_to_plot_per_population),
        len(candidate_local_positions),
    )
    if not n_plot:
        selected = events.iloc[0:0].copy()
    else:
        rng = np.random.default_rng(seed)
        selected_local_positions = rng.choice(
            candidate_local_positions, size=n_plot, replace=False
        )
        selected = events.iloc[
            event_positions[selected_local_positions]
        ].copy()
        selected["high_tail_qdc_mV_ns"] = qdc_values[
            selected_local_positions
        ]
        selected["high_tail_population"] = population
    return selected, threshold, effective_fraction, len(candidate_local_positions)
(
    high_qdc_pedestal_events, high_qdc_pedestal_threshold,
    high_qdc_pedestal_fraction, n_high_qdc_pedestal_candidates,
) = sample_high_qdc_tail(
    sampled_events, sampled_pedestal_positions, pedestal_qdc_values,
    "pedestal",
    sample_seed + 201,
)
(
    high_qdc_signal_events, high_qdc_signal_threshold,
    high_qdc_signal_fraction, n_high_qdc_signal_candidates,
) = sample_high_qdc_tail(
    sampled_events, sampled_peak_positions_after_mismatch_cut,
    peak_qdc_values, "signal", sample_seed + 202,
)
print(
    f"High-QDC pedestal tail: top {100 * high_qdc_pedestal_fraction:.3g}% "
    f"(QDC >= {high_qdc_pedestal_threshold:.6g} mV ns; "
    f"{n_high_qdc_pedestal_candidates:,} candidates)."
)
print(
    f"High-QDC signal tail: top {100 * high_qdc_signal_fraction:.3g}% "
    f"(QDC >= {high_qdc_signal_threshold:.6g} mV ns; "
    f"{n_high_qdc_signal_candidates:,} candidates)."
)
high_qdc_diagnostic_events = pd.concat(
    [high_qdc_pedestal_events, high_qdc_signal_events], ignore_index=True
)
if len(high_qdc_diagnostic_events):
    high_qdc_time_ns, high_qdc_diagnostic_waveforms_mV = load_event_waveforms(
        high_qdc_diagnostic_events,
        channel=sample_preprocessing["channel"],
        baseline_window_ns=sample_baseline_window_ns,
        baseline_reference_time_ns=reference_time_ns,
        baseline_reference_mV=reference_mV,
    )
    high_qdc_records = []
    for (_, event), waveform_mV in zip(
        high_qdc_diagnostic_events.iterrows(), high_qdc_diagnostic_waveforms_mV
    ):
        waveform_mV = np.asarray(waveform_mV, dtype=float)
        if event["high_tail_population"] == "pedestal":
            waveform_mV = waveform_mV - np.mean(waveform_mV)
        high_qdc_records.append((event, waveform_mV))

    n_high_qdc_rows = max(
        len(high_qdc_pedestal_events), len(high_qdc_signal_events)
    )
    fig_high_qdc, high_qdc_axes = plt.subplots(
        n_high_qdc_rows, 2, figsize=(15, 4.2 * n_high_qdc_rows),
        squeeze=False, sharex=True,
    )
    high_qdc_population_rows = {"pedestal": 0, "signal": 0}
    high_qdc_population_columns = {"pedestal": 0, "signal": 1}
    high_qdc_population_colors = {
        "pedestal": "tab:orange", "signal": "tab:blue"
    }
    for event, waveform_mV in high_qdc_records:
        population = event["high_tail_population"]
        row = high_qdc_population_rows[population]
        high_qdc_population_rows[population] += 1
        ax = high_qdc_axes[row, high_qdc_population_columns[population]]
        ax.plot(
            high_qdc_time_ns, waveform_mV,
            color=high_qdc_population_colors[population], linewidth=1.0,
        )
        ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.6)
        ax.axvspan(
            *sample_baseline_window_ns, color="0.5", alpha=0.08,
            label="baseline window",
        )
        if population == "pedestal":
            ax.axvspan(
                *pedestal_qdc_window_ns, color="tab:orange", alpha=0.07,
                label="pedestal QDC window",
            )
        else:
            peak_time = float(event.get("peak_time_ns", np.nan))
            peak_width = float(event.get("peak_width_ns", np.nan))
            if np.isfinite(peak_time):
                ax.axvline(
                    peak_time, color="tab:red", linestyle="--",
                    linewidth=1.1, label="accepted peak",
                )
            if np.isfinite(peak_time) and np.isfinite(peak_width):
                ax.axvspan(
                    peak_time - peak_width, peak_time + 2.0 * peak_width,
                    color="tab:blue", alpha=0.07, label="signal QDC window",
                )
        amplitude = float(event.get(
            "peak_amplitude_mV", event.get("max_excursion_amplitude_mV", np.nan)
        ))
        detail_lines = [f"QDC={float(event['high_tail_qdc_mV_ns']):.4g} mV ns"]
        if np.isfinite(amplitude):
            detail_lines.append(f"amplitude={amplitude:.4g} mV")
        snr = float(event.get("snr", np.nan))
        if np.isfinite(snr):
            detail_lines.append(f"SNR={snr:.3g}")
        fwhm = float(event.get("peak_width_ns", np.nan))
        if np.isfinite(fwhm):
            detail_lines.append(f"FWHM={fwhm:.3g} ns")
        details = "; ".join(detail_lines)
        ax.text(
            0.98, 0.04, details, transform=ax.transAxes, ha="right",
            va="bottom", fontsize=8,
            bbox={"facecolor": "white", "alpha": 0.78, "edgecolor": "0.7"},
        )
        ax.set_title(
            f"{population}: {Path(event['event_file']).name}, "
            f"segment {int(event['event_segment'])}", fontsize=9,
        )
        ax.set(xlabel="Time [ns]", ylabel="Voltage [mV]")
        ax.grid(True, which="both", alpha=0.35)
    for population, column in high_qdc_population_columns.items():
        for row in range(high_qdc_population_rows[population], n_high_qdc_rows):
            high_qdc_axes[row, column].set_visible(False)
    high_qdc_visible_axes = [
        ax for ax in high_qdc_axes.ravel() if ax.get_visible()
    ]
    legend_entries = {}
    for visible_ax in high_qdc_visible_axes:
        handles, labels = visible_ax.get_legend_handles_labels()
        legend_entries.update(
            (label, handle) for handle, label in zip(handles, labels) if label
        )
    fig_high_qdc.legend(
        legend_entries.values(), legend_entries.keys(),
        loc="lower center", ncols=4, fontsize=8,
    )
    fig_high_qdc.suptitle(
        f"{sample_acquisition}: representative waveforms from highest-QDC tails\n"
        f"pedestal top {100 * high_qdc_pedestal_fraction:.3g}% "
        f"(>= {high_qdc_pedestal_threshold:.4g}); signal top "
        f"{100 * high_qdc_signal_fraction:.3g}% "
        f"(>= {high_qdc_signal_threshold:.4g}) mV ns"
    )
    fig_high_qdc.tight_layout(rect=(0, 0.035, 1, 0.96))
    plt.show()

fig, (ax_waveforms, ax_qdc_populations, ax_qdc_combined) = plt.subplots(
    1, 3, figsize=(24, 7)
)
for index, record in enumerate(pedestal_plot_records):
    waveform = record["waveform_mV"]
    baseline_rms_mV = record["baseline_rms_mV"]
    ax_waveforms.plot(
        sample_time_ns, waveform, color="tab:orange", alpha=0.42, linewidth=0.9,
        label=(f"pedestal-like (shown {n_pedestal_waveforms_to_plot}/{n_pedestal_waveforms})" if index == 0 else None),
    )
    ax_waveforms.axhline(
        -sample_preprocessing["peak_snr_threshold"] * baseline_rms_mV,
        color="tab:orange", linestyle="--", alpha=0.08, linewidth=0.7,
    )
for index, record in enumerate(peak_plot_records):
    waveform = record["waveform_mV"]
    baseline_rms_mV = record["baseline_rms_mV"]
    metrics = record.get("accepted_peak_metrics") or record["metrics"]
    peak_time_ns = metrics["time_ns"]
    ax_waveforms.plot(
        sample_time_ns, waveform, color="tab:blue", alpha=0.52, linewidth=0.9,
        label=(
            f"post-mismatch signal (shown {len(peak_plot_records)}/"
            f"{n_peak_waveforms}; {signal_shape_survival_annotation})"
            if index == 0 else None
        ),
    )
    ax_waveforms.axvline(peak_time_ns, color="tab:blue", linestyle="--", alpha=0.18, linewidth=0.8)
    ax_waveforms.axhline(
        -sample_preprocessing["peak_snr_threshold"] * baseline_rms_mV,
        color="tab:blue", linestyle="--", alpha=0.08, linewidth=0.7,
    )
    width_left_ns = metrics["width_left_ns"]
    width_right_ns = metrics["width_right_ns"]
    width_level_mV = metrics["width_level_mV"]
    ax_waveforms.hlines(
        width_level_mV, width_left_ns, width_right_ns,
        color="tab:green", alpha=0.45, linewidth=2.0,
    )
ax_waveforms.axvspan(
    *sample_baseline_window_ns, color="0.5", alpha=0.10, label="original cache baseline window (peaks)"
)
ax_waveforms.axvspan(
    *sample_fixed_window_ns, color="tab:purple", alpha=0.10,
    label=f"fixed pulse charge window [{sample_fixed_window_ns[0]:.2f}, {sample_fixed_window_ns[1]:.2f}] ns",
)
ax_waveforms.axvspan(
    *pedestal_qdc_window_ns, color="tab:orange", alpha=0.06, hatch="//",
    label=f"pedestal QDC window [{pedestal_qdc_window_ns[0]:.0f}, {pedestal_qdc_window_ns[1]:.0f}] ns",
)
ax_waveforms.axhline(0.0, color="black", linewidth=0.8, alpha=0.6)
ax_waveforms.set(
    xlabel="Time [ns]", ylabel="Voltage [mV]",
    title=f"{sample_acquisition}: sampled pedestal and peak waveforms",
)
ax_waveforms.grid(True, which="both", alpha=0.35)
ax_waveforms.legend(fontsize=8, loc="lower right")

finite_qdc = combined_qdc_values[np.isfinite(combined_qdc_values)]
if not len(finite_qdc):
    raise ValueError("No finite QDC values were calculated")
qdc_min, qdc_max = float(np.min(finite_qdc)), float(np.max(finite_qdc))
if qdc_min == qdc_max:
    qdc_min -= 0.5
    qdc_max += 0.5
qdc_edges = np.linspace(qdc_min, qdc_max, qdc_histogram_bins + 1)
ax_qdc_populations.hist(
    pedestal_qdc_plot_values, bins=qdc_edges, density=qdc_histogram_density, histtype="step",
    weights=qdc_histogram_weights(pedestal_qdc_plot_values, "pedestal"),
    linewidth=2.0, color="tab:orange",
    label=f"pedestal 0-80 ns QDC (N={pedestal_qdc_plot_total:,})",
)
ax_qdc_populations.hist(
    peak_qdc_values, bins=qdc_edges, density=qdc_histogram_density, histtype="step",
    weights=qdc_histogram_weights(peak_qdc_values, "signal"),
    linewidth=2.0, color="tab:blue",
    label=(
        f"peak-centered QDC after mismatch cut (N={len(peak_qdc_values):,}; "
        f"survival={signal_mismatch_survival_percent:.2f}%)"
    ),
)
ax_qdc_populations.set(
    xlabel="QDC [mV ns]",
    ylabel=qdc_histogram_y_label,
    title=(
        "Separate QDC populations\n"
        "peaks: peak time - FWHM to peak time + 2 FWHM; "
        "pedestals: fixed 0-80 ns window"
    ),
)
ax_qdc_populations.grid(True, which="both", alpha=0.35)
ax_qdc_populations.legend(fontsize=9)

ax_qdc_combined.hist(
    combined_qdc_values, bins=qdc_edges, density=qdc_histogram_density, histtype="step",
    weights=combined_qdc_histogram_weights(
        pedestal_qdc_plot_values, peak_qdc_values
    ),
    linewidth=2.2, color="tab:purple", label=f"combined (N={len(combined_qdc_values)})",
)
ax_qdc_combined.set(
    xlabel="QDC [mV ns]",
    ylabel=qdc_histogram_y_label,
    title="Combined pedestal + peak QDC",
)
ax_qdc_combined.grid(True, which="both", alpha=0.35)
ax_qdc_combined.legend(fontsize=9)
fig.tight_layout()
plt.show()

# Same QDC distributions in a separate figure with logarithmic event counts.
fig_qdc_log, (ax_qdc_populations_log, ax_qdc_combined_log) = plt.subplots(
    1, 2, figsize=(15, 6)
)
ax_qdc_populations_log.hist(
    pedestal_qdc_plot_values, bins=qdc_edges, density=qdc_histogram_density,
    weights=qdc_histogram_weights(pedestal_qdc_plot_values, "pedestal"),
    histtype="step", linewidth=2.0, color="tab:orange",
    label=f"pedestal 0-80 ns QDC (N={pedestal_qdc_plot_total:,})",
)
ax_qdc_populations_log.hist(
    peak_qdc_values, bins=qdc_edges, density=qdc_histogram_density,
    weights=qdc_histogram_weights(peak_qdc_values, "signal"),
    histtype="step", linewidth=2.0, color="tab:blue",
    label=(
        f"peak-centered QDC after mismatch cut (N={len(peak_qdc_values):,}; "
        f"survival={signal_mismatch_survival_percent:.2f}%)"
    ),
)
ax_qdc_populations_log.set(
    xlabel="QDC [mV ns]",
    ylabel=qdc_histogram_y_label,
    title="Separate QDC populations — logarithmic y-axis",
    yscale="log",
)
ax_qdc_populations_log.grid(True, which="both", alpha=0.35)
ax_qdc_populations_log.legend(fontsize=9)

ax_qdc_combined_log.hist(
    combined_qdc_values, bins=qdc_edges, density=qdc_histogram_density,
    weights=combined_qdc_histogram_weights(
        pedestal_qdc_plot_values, peak_qdc_values
    ),
    histtype="step", linewidth=2.2, color="tab:purple",
    label=f"combined (N={len(combined_qdc_values)})",
)
ax_qdc_combined_log.set(
    xlabel="QDC [mV ns]",
    ylabel=qdc_histogram_y_label,
    title="Combined pedestal + peak QDC — logarithmic y-axis",
    yscale="log",
)
ax_qdc_combined_log.grid(True, which="both", alpha=0.35)
ax_qdc_combined_log.legend(fontsize=9)
fig_qdc_log.suptitle(
    "Event-by-event peak-centered QDC: "
    "peak time − FWHM to peak time + 2×FWHM"
)
fig_qdc_log.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

# Rise-time distribution and its relationship to the event-by-event QDC
# for exactly the events included in the blue signal population.
blue_rise_times_ns = sampled_peaks["rise_time_10_90_ns"].to_numpy(
    dtype=float
)
finite_blue_rise = np.isfinite(blue_rise_times_ns)
finite_blue_rise_qdc = finite_blue_rise & np.isfinite(peak_qdc_values)
if not np.any(finite_blue_rise):
    print("Skipping blue rise-time diagnostics: no finite rise times.")
else:
    fig_blue_rise, (ax_blue_rise_hist, ax_blue_rise_qdc) = plt.subplots(
        1, 2, figsize=(15, 6)
    )
    ax_blue_rise_hist.hist(
        blue_rise_times_ns[finite_blue_rise], bins=rise_time_histogram_bins,
        histtype="step", linewidth=2.0, color="tab:blue",
        label=(
            f"post-mismatch blue population (N={np.sum(finite_blue_rise):,}; "
            f"survival={signal_mismatch_survival_percent:.2f}%)"
        ),
    )
    ax_blue_rise_hist.set(
        xlabel="10–90% rise time [ns]", ylabel="Waveforms",
        title="Blue-population 10–90% rise-time distribution",
    )
    ax_blue_rise_hist.grid(True, which="both", alpha=0.35)
    ax_blue_rise_hist.legend(fontsize=9)
    if np.any(finite_blue_rise_qdc):
        rise_qdc_hist = ax_blue_rise_qdc.hist2d(
            blue_rise_times_ns[finite_blue_rise_qdc],
            peak_qdc_values[finite_blue_rise_qdc],
            bins=rise_time_qdc_density_bins, cmap="jet", norm=LogNorm(),
        )
        fig_blue_rise.colorbar(
            rise_qdc_hist[3], ax=ax_blue_rise_qdc, label="Waveforms per bin"
        )
    ax_blue_rise_qdc.set(
        xlabel="10–90% rise time [ns]",
        ylabel="Event-by-event peak-centered QDC [mV ns]",
        title="Blue-population rise time vs event-by-event QDC",
    )
    ax_blue_rise_qdc.grid(True, which="both", alpha=0.25)
    fig_blue_rise.tight_layout()
    plt.show()

# Direct comparison using the learned fixed integration window for both
# populations. These charges were calculated and cached by Selection.
fixed_charge_column = sample_fixed_window.get(
    "column", "charge_fixed_pulse_window_mV_ns"
)
if not sample_fixed_charge_available:
    print(
        f"Skipping fixed-window QDC comparison: cache has no "
        f"{fixed_charge_column!r} column."
    )
else:
    fixed_pedestal_qdc = sampled_pedestals[fixed_charge_column].to_numpy(
        dtype=float
    )
    fixed_signal_qdc = sampled_peaks[fixed_charge_column].to_numpy(dtype=float)
    fixed_pedestal_qdc = fixed_pedestal_qdc[np.isfinite(fixed_pedestal_qdc)]
    fixed_signal_qdc = fixed_signal_qdc[np.isfinite(fixed_signal_qdc)]
    fixed_combined_qdc = np.concatenate([fixed_pedestal_qdc, fixed_signal_qdc])
    if not len(fixed_combined_qdc):
        print("Skipping fixed-window QDC comparison: no finite charges.")
    else:
        fixed_qdc_min = float(np.min(fixed_combined_qdc))
        fixed_qdc_max = float(np.max(fixed_combined_qdc))
        if fixed_qdc_min == fixed_qdc_max:
            fixed_qdc_min -= 0.5
            fixed_qdc_max += 0.5
        fixed_qdc_edges = np.linspace(
            fixed_qdc_min, fixed_qdc_max, qdc_histogram_bins + 1
        )
        fig_fixed_qdc_log, (
            ax_fixed_qdc_populations_log, ax_fixed_qdc_combined_log
        ) = plt.subplots(1, 2, figsize=(15, 6))
        ax_fixed_qdc_populations_log.hist(
            fixed_pedestal_qdc, bins=fixed_qdc_edges,
            density=qdc_histogram_density, histtype="step", linewidth=2.0,
            weights=qdc_histogram_weights(fixed_pedestal_qdc, "pedestal"),
            color="tab:orange",
            label=f"pedestal fixed-window QDC (N={len(fixed_pedestal_qdc):,})",
        )
        ax_fixed_qdc_populations_log.hist(
            fixed_signal_qdc, bins=fixed_qdc_edges,
            density=qdc_histogram_density, histtype="step", linewidth=2.0,
            weights=qdc_histogram_weights(fixed_signal_qdc, "signal"),
            color="tab:blue",
            label=(
                f"post-mismatch signal fixed-window QDC "
                f"(N={len(fixed_signal_qdc):,}; "
                f"survival={signal_mismatch_survival_percent:.2f}%)"
            ),
        )
        ax_fixed_qdc_populations_log.set(
            xlabel="QDC [mV ns]",
            ylabel=qdc_histogram_y_label,
            title="Separate populations — learned fixed integration window",
            yscale="log",
        )
        ax_fixed_qdc_populations_log.grid(True, which="both", alpha=0.35)
        ax_fixed_qdc_populations_log.legend(fontsize=9)
        ax_fixed_qdc_combined_log.hist(
            fixed_combined_qdc, bins=fixed_qdc_edges,
            density=qdc_histogram_density, histtype="step", linewidth=2.2,
            weights=combined_qdc_histogram_weights(
                fixed_pedestal_qdc, fixed_signal_qdc
            ),
            color="tab:purple",
            label=f"combined fixed-window QDC (N={len(fixed_combined_qdc):,})",
        )
        ax_fixed_qdc_combined_log.set(
            xlabel="QDC [mV ns]",
            ylabel=qdc_histogram_y_label,
            title="Combined population — learned fixed integration window",
            yscale="log",
        )
        ax_fixed_qdc_combined_log.grid(True, which="both", alpha=0.35)
        ax_fixed_qdc_combined_log.legend(fontsize=9)
        fig_fixed_qdc_log.suptitle(
            f"Fixed-window QDC comparison: {sample_fixed_window_ns[0]:.3g}–"
            f"{sample_fixed_window_ns[1]:.3g} ns for every event"
        )
        fig_fixed_qdc_log.tight_layout(rect=(0, 0, 1, 0.95))
        plt.show()

# Raw full-trace comparison: integrate -V_raw over the complete recorded
# waveform with no baseline subtraction. Only the same pedestal and final
# blue-signal populations used above are included.
raw_full_pedestal_qdc = np.asarray(
    pedestal_raw_full_waveform_qdc_values, dtype=float
)
raw_full_signal_qdc = np.asarray(
    peak_raw_full_waveform_qdc_values, dtype=float
)
raw_full_pedestal_qdc = raw_full_pedestal_qdc[
    np.isfinite(raw_full_pedestal_qdc)
]
raw_full_signal_qdc = raw_full_signal_qdc[np.isfinite(raw_full_signal_qdc)]
raw_full_combined_qdc = np.concatenate([
    raw_full_pedestal_qdc, raw_full_signal_qdc
])
if not len(raw_full_combined_qdc):
    print("Skipping raw full-trace QDC comparison: no finite charges.")
else:
    raw_full_qdc_min = float(np.min(raw_full_combined_qdc))
    raw_full_qdc_max = float(np.max(raw_full_combined_qdc))
    if raw_full_qdc_min == raw_full_qdc_max:
        raw_full_qdc_min -= 0.5
        raw_full_qdc_max += 0.5
    raw_full_qdc_edges = np.linspace(
        raw_full_qdc_min, raw_full_qdc_max, qdc_histogram_bins + 1
    )
    fig_raw_full_qdc_log, (
        ax_raw_full_populations_log, ax_raw_full_combined_log
    ) = plt.subplots(1, 2, figsize=(15, 6))
    ax_raw_full_populations_log.hist(
        raw_full_pedestal_qdc, bins=raw_full_qdc_edges,
        density=qdc_histogram_density, histtype="step", linewidth=2.0,
        weights=qdc_histogram_weights(raw_full_pedestal_qdc, "pedestal"),
        color="tab:orange",
        label=f"pedestal raw full trace (N={len(raw_full_pedestal_qdc):,})",
    )
    ax_raw_full_populations_log.hist(
        raw_full_signal_qdc, bins=raw_full_qdc_edges,
        density=qdc_histogram_density, histtype="step", linewidth=2.0,
        weights=qdc_histogram_weights(raw_full_signal_qdc, "signal"),
        color="tab:blue",
        label=(
            f"post-mismatch signal raw full trace "
            f"(N={len(raw_full_signal_qdc):,}; "
            f"survival={signal_mismatch_survival_percent:.2f}%)"
        ),
    )
    ax_raw_full_populations_log.set(
        xlabel="Raw full-trace integral [mV ns]",
        ylabel=qdc_histogram_y_label,
        title="Separate accepted populations — no baseline subtraction",
        yscale="log",
    )
    ax_raw_full_populations_log.grid(True, which="both", alpha=0.35)
    ax_raw_full_populations_log.legend(fontsize=9)
    ax_raw_full_combined_log.hist(
        raw_full_combined_qdc, bins=raw_full_qdc_edges,
        density=qdc_histogram_density, histtype="step", linewidth=2.2,
        weights=combined_qdc_histogram_weights(
            raw_full_pedestal_qdc, raw_full_signal_qdc
        ),
        color="tab:purple",
        label=f"combined raw full trace (N={len(raw_full_combined_qdc):,})",
    )
    ax_raw_full_combined_log.set(
        xlabel="Raw full-trace integral [mV ns]",
        ylabel=qdc_histogram_y_label,
        title="Combined accepted population — no baseline subtraction",
        yscale="log",
    )
    ax_raw_full_combined_log.grid(True, which="both", alpha=0.35)
    ax_raw_full_combined_log.legend(fontsize=9)
    fig_raw_full_qdc_log.suptitle(
        "Raw full-waveform integration: −∫V_raw dt over the entire trace"
    )
    fig_raw_full_qdc_log.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()

# Compare the three integration methods as binary pedestal/signal
# discriminators. The waveform classification is the reference label here;
# this is not an independently measured physical ground truth.
def qdc_valley_classification(pedestal_values, signal_values):
    pedestal_values = np.asarray(pedestal_values, dtype=float)
    signal_values = np.asarray(signal_values, dtype=float)
    pedestal_values = pedestal_values[np.isfinite(pedestal_values)]
    signal_values = signal_values[np.isfinite(signal_values)]
    if not len(pedestal_values) or not len(signal_values):
        return None
    n_bins = int(qdc_separator_histogram_bins)
    smoothing_sigma = float(qdc_separator_smoothing_sigma_bins)
    range_low_q, range_high_q = map(
        float, qdc_separator_range_quantiles
    )
    if n_bins < 8:
        raise ValueError("qdc_separator_histogram_bins must be at least 8")
    if smoothing_sigma <= 0.0:
        raise ValueError(
            "qdc_separator_smoothing_sigma_bins must be positive"
        )
    if not 0.0 <= range_low_q < range_high_q <= 1.0:
        raise ValueError(
            "qdc_separator_range_quantiles must satisfy "
            "0 <= low < high <= 1"
        )
    combined_values = np.concatenate([pedestal_values, signal_values])
    histogram_low, histogram_high = np.quantile(
        combined_values, (range_low_q, range_high_q)
    )
    histogram_low = float(histogram_low)
    histogram_high = float(histogram_high)
    if histogram_low == histogram_high:
        padding = max(0.5, abs(histogram_low) * 0.01)
        histogram_low -= padding
        histogram_high += padding
    edges = np.linspace(histogram_low, histogram_high, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    pedestal_counts, _ = np.histogram(pedestal_values, bins=edges)
    signal_counts, _ = np.histogram(signal_values, bins=edges)

    kernel_radius = max(1, int(np.ceil(4.0 * smoothing_sigma)))
    kernel_offsets = np.arange(-kernel_radius, kernel_radius + 1)
    smoothing_kernel = np.exp(
        -0.5 * (kernel_offsets / smoothing_sigma) ** 2
    )
    smoothing_kernel /= smoothing_kernel.sum()

    def smooth_counts(counts):
        return np.convolve(
            np.asarray(counts, dtype=float), smoothing_kernel, mode="same"
        )

    pedestal_smoothed_counts = smooth_counts(pedestal_counts)
    signal_smoothed_counts = smooth_counts(signal_counts)
    pedestal_mode_index = int(np.argmax(pedestal_smoothed_counts))
    signal_mode_index = int(np.argmax(signal_smoothed_counts))
    if pedestal_mode_index >= signal_mode_index:
        # A noisy mode estimate can invert even when the population centers
        # are ordered. Median anchors retain the requested left/right rule.
        pedestal_median = float(np.median(pedestal_values))
        signal_median = float(np.median(signal_values))
        if pedestal_median >= signal_median:
            return None
        pedestal_mode_index = int(np.clip(
            np.searchsorted(centers, pedestal_median), 0, n_bins - 1
        ))
        signal_mode_index = int(np.clip(
            np.searchsorted(centers, signal_median), 0, n_bins - 1
        ))
    valley_indices = np.arange(
        pedestal_mode_index + 1, signal_mode_index
    )
    if not len(valley_indices):
        valley_indices = np.arange(
            pedestal_mode_index, signal_mode_index + 1
        )

    if qdc_separator_balance_classes:
        pedestal_density = (
            pedestal_smoothed_counts / max(pedestal_counts.sum(), 1)
        )
        signal_density = (
            signal_smoothed_counts / max(signal_counts.sum(), 1)
        )
        combined_for_valley = pedestal_density + signal_density
    else:
        pedestal_density = pedestal_smoothed_counts
        signal_density = signal_smoothed_counts
        combined_for_valley = smooth_counts(
            pedestal_counts + signal_counts
        )
    valley_index = int(valley_indices[
        np.argmin(combined_for_valley[valley_indices])
    ])
    separator = float(centers[valley_index])

    true_negative = int(np.sum(pedestal_values <= separator))
    false_positive = int(np.sum(pedestal_values > separator))
    false_negative = int(np.sum(signal_values <= separator))
    true_positive = int(np.sum(signal_values > separator))
    true_negative_rate = true_negative / len(pedestal_values)
    false_positive_rate = false_positive / len(pedestal_values)
    true_positive_rate = true_positive / len(signal_values)
    false_negative_rate = false_negative / len(signal_values)
    balanced_accuracy = 0.5 * (
        true_positive_rate + true_negative_rate
    )
    return {
        "separator_mV_ns": separator,
        "edges": edges,
        "centers": centers,
        "pedestal_counts": pedestal_counts,
        "signal_counts": signal_counts,
        "combined_for_valley": combined_for_valley,
        "pedestal_mode_mV_ns": float(centers[pedestal_mode_index]),
        "signal_mode_mV_ns": float(centers[signal_mode_index]),
        "TP": true_positive, "TN": true_negative,
        "FP": false_positive, "FN": false_negative,
        "TPR": true_positive_rate, "TNR": true_negative_rate,
        "FPR": false_positive_rate, "FNR": false_negative_rate,
        "balanced_accuracy": balanced_accuracy,
        "n_pedestal": len(pedestal_values),
        "n_signal": len(signal_values),
    }

qdc_separation_method_populations = {
    "event-specific": (pedestal_qdc_plot_values, peak_qdc_values),
    "raw full trace": (raw_full_pedestal_qdc, raw_full_signal_qdc),
}
if sample_fixed_charge_available:
    qdc_separation_method_populations = {
        "fixed window": (fixed_pedestal_qdc, fixed_signal_qdc),
        **qdc_separation_method_populations,
    }
qdc_separation_results = {
    method: qdc_valley_classification(pedestal_values, signal_values)
    for method, (pedestal_values, signal_values)
    in qdc_separation_method_populations.items()
}
qdc_separation_results = {
    method: result for method, result in qdc_separation_results.items()
    if result is not None
}
if not qdc_separation_results:
    print(
        "Skipping QDC separation confusion plot: both pedestal and signal "
        "populations are required."
    )
else:
    print("QDC valley separation relative to waveform-based labels:")
    for method, result in qdc_separation_results.items():
        print(
            f"  {method}: separator={result['separator_mV_ns']:.6g} mV ns; "
            f"TP={result['TP']:,}, TN={result['TN']:,}, "
            f"FP={result['FP']:,}, FN={result['FN']:,}; "
            f"balanced accuracy={100 * result['balanced_accuracy']:.2f}%"
        )

    separation_methods = list(qdc_separation_results)
    fig_qdc_separation = plt.figure(figsize=(18, 11))
    separation_grid = fig_qdc_separation.add_gridspec(
        2, len(separation_methods), height_ratios=(1.05, 1.0)
    )
    for column, method in enumerate(separation_methods):
        result = qdc_separation_results[method]
        ax = fig_qdc_separation.add_subplot(separation_grid[0, column])
        edges = result["edges"]
        ax.stairs(
            result["pedestal_counts"], edges, color="tab:orange",
            linewidth=1.4, alpha=0.75, label="pedestal label",
        )
        ax.stairs(
            result["signal_counts"], edges, color="tab:blue",
            linewidth=1.4, alpha=0.75, label="signal label",
        )
        ax.stairs(
            result["pedestal_counts"] + result["signal_counts"],
            edges, color="tab:purple", linewidth=2.0, alpha=0.7,
            label="combined counts",
        )
        ax.axvline(
            result["separator_mV_ns"], color="black", linestyle="--",
            linewidth=1.6,
            label=f"valley={result['separator_mV_ns']:.4g} mV ns",
        )
        ax.set(
            xlabel="QDC [mV ns]", ylabel="Events per bin",
            title=(
                f"{method}\n"
                f"balanced accuracy={100 * result['balanced_accuracy']:.2f}%"
            ),
            yscale="log",
        )
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)

    ax_confusion = fig_qdc_separation.add_subplot(separation_grid[1, :])
    confusion_categories = ("TP", "TN", "FP", "FN")
    confusion_rate_keys = ("TPR", "TNR", "FPR", "FNR")
    confusion_colors = (
        "tab:blue", "tab:orange", "tab:red", "tab:purple"
    )
    method_positions = np.arange(len(separation_methods), dtype=float)
    bar_width = 0.19
    for category_index, (category, rate_key, color) in enumerate(zip(
        confusion_categories, confusion_rate_keys, confusion_colors
    )):
        offsets = (category_index - 1.5) * bar_width
        rates_percent = [
            100.0 * qdc_separation_results[method][rate_key]
            for method in separation_methods
        ]
        bars = ax_confusion.bar(
            method_positions + offsets, rates_percent, width=bar_width,
            color=color, alpha=0.75, label=category,
        )
        raw_counts = [
            qdc_separation_results[method][category]
            for method in separation_methods
        ]
        for bar, rate_percent, raw_count in zip(
            bars, rates_percent, raw_counts
        ):
            annotation_inside = rate_percent >= 12.0
            ax_confusion.annotate(
                f"{rate_percent:.2f}%\nN={raw_count:,}",
                (bar.get_x() + bar.get_width() / 2.0, bar.get_height()),
                xytext=(0, -4 if annotation_inside else 3),
                textcoords="offset points", ha="center",
                va="top" if annotation_inside else "bottom",
                fontsize=8, rotation=90,
            )
    best_method = max(
        separation_methods,
        key=lambda method: qdc_separation_results[method][
            "balanced_accuracy"
        ],
    )
    method_tick_labels = [
        f"{method}\nseparator={qdc_separation_results[method]['separator_mV_ns']:.4g}"
        f"\nBA={100 * qdc_separation_results[method]['balanced_accuracy']:.2f}%"
        for method in separation_methods
    ]
    ax_confusion.set(
        xticks=method_positions, xticklabels=method_tick_labels,
        ylabel="Rate within reference class [%]", ylim=(0.0, 105.0),
        title=(
            "QDC-side confusion rates (signal is positive; left is pedestal)\n"
            "TP/FN normalized by signals; TN/FP normalized by pedestals"
        ),
    )
    ax_confusion.grid(True, axis="y", alpha=0.3)
    ax_confusion.legend(ncol=4, fontsize=9, loc="center right")
    balancing_description = (
        "class-balanced smoothed density"
        if qdc_separator_balance_classes else "raw combined counts"
    )
    fig_qdc_separation.suptitle(
        f"{sample_acquisition}: automatic QDC valley separators\n"
        f"separator from {balancing_description}; best balanced accuracy: "
        f"{best_method} "
        f"({100 * qdc_separation_results[best_method]['balanced_accuracy']:.2f}%)\n"
        "Reference truth is the waveform-based pedestal/signal classification",
        fontsize=14,
    )
    fig_qdc_separation.tight_layout(rect=(0, 0, 1, 0.91))
    plt.show()

# Compare pedestal charge windows. Full-waveform charge uses baseline
# sidebands outside the expected pulse region; subtracting the full-trace
# mean would force the full-trace integral to zero by construction.
finite_pedestal_window_qdc = pedestal_qdc_values[
    np.isfinite(pedestal_qdc_values)
]
finite_pedestal_full_qdc = pedestal_full_waveform_qdc_values[
    np.isfinite(pedestal_full_waveform_qdc_values)
]
pedestal_charge_comparison = np.concatenate([
    pedestal_qdc_plot_values, pedestal_full_qdc_plot_values
])
if len(pedestal_charge_comparison):
    pedestal_charge_lo = float(np.min(pedestal_charge_comparison))
    pedestal_charge_hi = float(np.max(pedestal_charge_comparison))
    if pedestal_charge_lo == pedestal_charge_hi:
        pedestal_charge_lo -= 0.5
        pedestal_charge_hi += 0.5
else:
    pedestal_charge_lo, pedestal_charge_hi = -0.5, 0.5
pedestal_charge_edges = np.linspace(
    pedestal_charge_lo, pedestal_charge_hi, qdc_histogram_bins + 1
)
fig_pedestal_charge, (
    ax_pedestal_charge_linear, ax_pedestal_charge_log
) = plt.subplots(1, 2, figsize=(15, 6), sharex=True)
for ax in (ax_pedestal_charge_linear, ax_pedestal_charge_log):
    ax.hist(
        pedestal_qdc_plot_values, bins=pedestal_charge_edges,
        density=qdc_histogram_density, histtype="step", linewidth=2.0,
        weights=qdc_histogram_weights(pedestal_qdc_plot_values, "pedestal"),
        color="tab:orange", label="pedestal 0-80 ns; full-trace-mean baseline",
    )
    ax.hist(
        pedestal_full_qdc_plot_values, bins=pedestal_charge_edges,
        density=qdc_histogram_density, histtype="step", linewidth=2.0,
        weights=qdc_histogram_weights(
            pedestal_full_qdc_plot_values, "pedestal"
        ),
        color="tab:purple",
        label=f"pedestal full waveform; sidebands {pedestal_full_waveform_baseline_sidebands_ns}",
    )
    ax.axvline(
        pedestal_qdc_tail_threshold_mV_ns, color="tab:red", linestyle="--",
        label=f"tail threshold = {pedestal_qdc_tail_threshold_mV_ns:g} mV ns",
    )
    ax.set(
        xlabel="Pedestal-like QDC [mV ns]",
        ylabel=qdc_histogram_y_label,
    )
    ax.grid(True, which="both", alpha=0.35)
    ax.legend(fontsize=8)
ax_pedestal_charge_linear.set_title("Pedestal charge-window comparison")
ax_pedestal_charge_log.set(
    title="Pedestal charge-window comparison — logarithmic y-axis",
    yscale="log",
)
fig_pedestal_charge.suptitle("All finite pedestal events included")
fig_pedestal_charge.tight_layout()
plt.show()

# Separate and combined populations using sideband-baselined full-waveform
# pedestal charge. Peak events retain their peak-centered QDC definition.
finite_peak_centered_qdc = np.asarray(peak_qdc_values, dtype=float)
finite_peak_centered_qdc = finite_peak_centered_qdc[
    np.isfinite(finite_peak_centered_qdc)
]
combined_full_pedestal_qdc_values = np.concatenate([
    pedestal_full_qdc_plot_values, finite_peak_centered_qdc
])
finite_combined_full_pedestal_qdc = combined_full_pedestal_qdc_values[
    np.isfinite(combined_full_pedestal_qdc_values)
]
full_population_edges = np.linspace(
    float(np.min(finite_combined_full_pedestal_qdc)),
    float(np.max(finite_combined_full_pedestal_qdc)),
    qdc_histogram_bins + 1,
)
fig_full_population, (
    ax_full_population_separate, ax_full_population_combined
) = plt.subplots(1, 2, figsize=(15, 6))
ax_full_population_separate.hist(
    pedestal_full_qdc_plot_values, bins=full_population_edges,
    density=qdc_histogram_density, histtype="step", linewidth=2.0,
    weights=qdc_histogram_weights(
        pedestal_full_qdc_plot_values, "pedestal"
    ),
    color="tab:purple",
    label=f"full-waveform pedestal (N={pedestal_full_qdc_plot_total:,})",
)
ax_full_population_separate.hist(
    finite_peak_centered_qdc, bins=full_population_edges,
    density=qdc_histogram_density, histtype="step", linewidth=2.0,
    weights=qdc_histogram_weights(finite_peak_centered_qdc, "signal"),
    color="tab:blue",
    label=(
        f"post-mismatch peak-centered QDC "
        f"(N={len(finite_peak_centered_qdc):,}; "
        f"survival={signal_mismatch_survival_percent:.2f}%)"
    ),
)
ax_full_population_separate.set(
    xlabel="QDC [mV ns]",
    ylabel=qdc_histogram_y_label,
    title="Separate populations: full-waveform pedestal QDC",
)
ax_full_population_separate.grid(True, which="both", alpha=0.35)
ax_full_population_separate.legend(fontsize=8)
ax_full_population_combined.hist(
    finite_combined_full_pedestal_qdc, bins=full_population_edges,
    density=qdc_histogram_density, histtype="step", linewidth=2.2,
    weights=combined_qdc_histogram_weights(
        pedestal_full_qdc_plot_values, finite_peak_centered_qdc
    ),
    color="tab:green",
    label=f"combined (N={len(finite_combined_full_pedestal_qdc):,})",
)
ax_full_population_combined.set(
    xlabel="QDC [mV ns]",
    ylabel=qdc_histogram_y_label,
    title="Combined: full-waveform pedestal + peak-centered signal",
)
ax_full_population_combined.grid(True, which="both", alpha=0.35)
ax_full_population_combined.legend(fontsize=8)
fig_full_population.suptitle("All finite pedestal events included")
fig_full_population.tight_layout()
plt.show()

# Maximum polarity-corrected excursion amplitudes: each population and combined.
finite_pedestal_amplitudes_mV = pedestal_max_amplitudes_mV[
    np.isfinite(pedestal_max_amplitudes_mV)
]
finite_peak_amplitudes_mV = peak_max_amplitudes_mV[
    np.isfinite(peak_max_amplitudes_mV)
]
combined_max_amplitudes_mV = np.concatenate([
    finite_pedestal_amplitudes_mV, finite_peak_amplitudes_mV
])
if not len(combined_max_amplitudes_mV):
    raise ValueError("No finite maximum amplitudes are available")
amplitude_edges = np.histogram_bin_edges(
    combined_max_amplitudes_mV, bins=amplitude_histogram_bins
)
fig_amplitude, (
    ax_pedestal_amplitude, ax_peak_amplitude, ax_combined_amplitude
) = plt.subplots(1, 3, figsize=(21, 6), sharex=True)
ax_pedestal_amplitude.hist(
    finite_pedestal_amplitudes_mV, bins=amplitude_edges,
    histtype="step", linewidth=2.0, color="tab:orange",
)
ax_pedestal_amplitude.set(
    xlabel="Maximum negative-pulse amplitude [mV]", ylabel="Waveforms",
    title=f"Pedestal-like maximum amplitude (N={len(pedestal_max_amplitudes_mV):,})",
)
ax_pedestal_amplitude.grid(True, which="both", alpha=0.35)

ax_peak_amplitude.hist(
    finite_peak_amplitudes_mV, bins=amplitude_edges,
    histtype="step", linewidth=2.0, color="tab:blue",
)
ax_peak_amplitude.axvline(
    sample_peak_amplitude_max_mV, color="tab:red", linestyle="--",
    label=f"sampling maximum = {sample_peak_amplitude_max_mV:g} mV",
)
ax_peak_amplitude.set(
    xlabel="Maximum negative-pulse amplitude [mV]", ylabel="Waveforms",
    title=(
        f"Post-mismatch signal maximum amplitude "
        f"(N={len(peak_max_amplitudes_mV):,}; "
        f"survival={signal_mismatch_survival_percent:.2f}%)"
    ),
)
ax_peak_amplitude.grid(True, which="both", alpha=0.35)
ax_peak_amplitude.legend()

ax_combined_amplitude.hist(
    combined_max_amplitudes_mV, bins=amplitude_edges,
    histtype="step", linewidth=2.2, color="tab:purple",
    label=f"combined (N={len(combined_max_amplitudes_mV):,})",
)
ax_combined_amplitude.axvline(
    sample_peak_amplitude_max_mV, color="tab:red", linestyle="--",
    label=f"peak sampling maximum = {sample_peak_amplitude_max_mV:g} mV",
)
ax_combined_amplitude.set(
    xlabel="Maximum negative-pulse amplitude [mV]", ylabel="Waveforms",
    title="Combined pedestal-like + classified-signal maximum amplitude",
)
ax_combined_amplitude.grid(True, which="both", alpha=0.35)
ax_combined_amplitude.legend(fontsize=9)
fig_amplitude.suptitle(
    "Maximum waveform excursion after population-specific baseline subtraction"
)
fig_amplitude.tight_layout()
plt.show()

# Maximum amplitude versus QDC, separated by cached population.
def finite_downsampled_xy(x_values, y_values, max_points):
    x_values = np.asarray(x_values, dtype=float)
    y_values = np.asarray(y_values, dtype=float)
    finite = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = x_values[finite]
    y_values = y_values[finite]
    if max_points is not None and len(x_values) > max_points:
        indices = np.linspace(0, len(x_values) - 1, max_points, dtype=int)
        x_values = x_values[indices]
        y_values = y_values[indices]
    return x_values, y_values, int(np.sum(finite))

pedestal_qdc_plot, pedestal_amplitude_plot, pedestal_xy_total = finite_downsampled_xy(
    pedestal_qdc_values, pedestal_max_amplitudes_mV, amplitude_qdc_scatter_max_points
)
peak_qdc_plot, peak_amplitude_plot, peak_xy_total = finite_downsampled_xy(
    peak_qdc_values, peak_max_amplitudes_mV, amplitude_qdc_scatter_max_points
)

# Assign each displayed peak point the count in its local 2D bin.  The bin
# counts use the full finite peak population, not only the displayed subset.
finite_peak_xy = np.isfinite(peak_qdc_values) & np.isfinite(peak_max_amplitudes_mV)
if not np.any(finite_peak_xy):
    raise ValueError("No finite peak amplitude/QDC pairs are available")
peak_density_hist, peak_qdc_edges, peak_amplitude_edges = np.histogram2d(
    np.asarray(peak_qdc_values)[finite_peak_xy],
    np.asarray(peak_max_amplitudes_mV)[finite_peak_xy],
    bins=amplitude_qdc_density_bins,
)
peak_qdc_bin = np.clip(
    np.searchsorted(peak_qdc_edges, peak_qdc_plot, side="right") - 1,
    0, len(peak_qdc_edges) - 2,
)
peak_amplitude_bin = np.clip(
    np.searchsorted(peak_amplitude_edges, peak_amplitude_plot, side="right") - 1,
    0, len(peak_amplitude_edges) - 2,
)
peak_local_density = peak_density_hist[peak_qdc_bin, peak_amplitude_bin]
fig_amplitude_qdc, (ax_pedestal_amplitude_qdc, ax_peak_amplitude_qdc) = plt.subplots(
    1, 2, figsize=(15, 6)
)
ax_pedestal_amplitude_qdc.scatter(
    pedestal_qdc_plot, pedestal_amplitude_plot, s=7, alpha=0.25,
    color="tab:orange", edgecolors="none",
)
ax_pedestal_amplitude_qdc.axvline(
    pedestal_qdc_tail_threshold_mV_ns, color="tab:red", linestyle="--",
    label=f"QDC threshold = {pedestal_qdc_tail_threshold_mV_ns:g} mV ns",
)
ax_pedestal_amplitude_qdc.set(
    xlabel="Pedestal-like QDC [mV ns]",
    ylabel="Maximum negative-pulse amplitude [mV]",
    title=(
        f"Pedestal-like amplitude vs QDC "
        f"(shown {len(pedestal_qdc_plot):,}/{pedestal_xy_total:,})"
    ),
)
ax_pedestal_amplitude_qdc.grid(True, which="both", alpha=0.35)
ax_pedestal_amplitude_qdc.legend()

peak_density_scatter = ax_peak_amplitude_qdc.scatter(
    peak_qdc_plot, peak_amplitude_plot, s=7, alpha=0.25,
    c=peak_local_density, cmap="jet",
    norm=LogNorm(vmin=1, vmax=max(2, float(np.max(peak_local_density)))),
    edgecolors="none",
)
peak_density_colorbar = fig_amplitude_qdc.colorbar(
    peak_density_scatter, ax=ax_peak_amplitude_qdc, pad=0.02
)
peak_density_colorbar.set_label("Peak population per 2D bin (log color scale)")
ax_peak_amplitude_qdc.axhline(
    sample_peak_amplitude_max_mV, color="tab:red", linestyle="--",
    label=f"sampling maximum = {sample_peak_amplitude_max_mV:g} mV",
)
ax_peak_amplitude_qdc.set(
    xlabel="Peak-centered QDC [mV ns]",
    ylabel="Maximum negative-pulse amplitude [mV]",
    title=(
        f"Post-mismatch signal amplitude vs QDC "
        f"(shown {len(peak_qdc_plot):,}/{peak_xy_total:,}; "
        f"survival={signal_mismatch_survival_percent:.2f}%)"
    ),
)
ax_peak_amplitude_qdc.grid(True, which="both", alpha=0.35)
ax_peak_amplitude_qdc.legend()
fig_amplitude_qdc.tight_layout()
plt.show()

# Empirical CDF and complementary CDF for pedestal-classified QDC values.
finite_pedestal_qdc = pedestal_qdc_values[np.isfinite(pedestal_qdc_values)]
if not len(finite_pedestal_qdc):
    print("Skipping pedestal QDC CDF statistics: no pedestal events are available.")
pedestal_cdf_denominator = max(len(finite_pedestal_qdc), 1)
sorted_pedestal_qdc = np.sort(finite_pedestal_qdc)
pedestal_cdf = np.arange(1, len(sorted_pedestal_qdc) + 1) / pedestal_cdf_denominator
pedestal_survival = (
    len(sorted_pedestal_qdc) - np.arange(len(sorted_pedestal_qdc))
) / pedestal_cdf_denominator
n_pedestal_above_threshold = int(np.sum(
    finite_pedestal_qdc > pedestal_qdc_tail_threshold_mV_ns
))
pedestal_fraction_above_threshold = (
    n_pedestal_above_threshold / pedestal_cdf_denominator
)

fig_pedestal_cdf, (ax_pedestal_cdf, ax_pedestal_survival) = plt.subplots(
    1, 2, figsize=(15, 6)
)
ax_pedestal_cdf.step(
    sorted_pedestal_qdc, 100 * pedestal_cdf, where="post",
    color="tab:orange", linewidth=1.8,
)
ax_pedestal_cdf.axvline(
    pedestal_qdc_tail_threshold_mV_ns, color="tab:red", linestyle="--",
    label=f"threshold = {pedestal_qdc_tail_threshold_mV_ns:g} mV ns",
)
ax_pedestal_cdf.set(
    xlabel="Pedestal-like QDC [mV ns]", ylabel="CDF [%]",
    title="Pedestal-like QDC empirical CDF", ylim=(0, 100),
)
ax_pedestal_cdf.grid(True, which="both", alpha=0.35)
ax_pedestal_cdf.legend()

ax_pedestal_survival.step(
    sorted_pedestal_qdc, 100 * pedestal_survival, where="post",
    color="tab:purple", linewidth=1.8,
)
ax_pedestal_survival.axvline(
    pedestal_qdc_tail_threshold_mV_ns, color="tab:red", linestyle="--",
)
ax_pedestal_survival.scatter(
    [pedestal_qdc_tail_threshold_mV_ns],
    [100 * pedestal_fraction_above_threshold],
    color="tab:red", edgecolor="black", zorder=5,
)
ax_pedestal_survival.text(
    0.97, 0.95,
    (
        f"QDC > {pedestal_qdc_tail_threshold_mV_ns:g} mV ns\n"
        f"{n_pedestal_above_threshold:,}/{len(finite_pedestal_qdc):,} events\n"
        f"{100 * pedestal_fraction_above_threshold:.3f}%"
    ),
    transform=ax_pedestal_survival.transAxes, ha="right", va="top",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
)
ax_pedestal_survival.set(
    xlabel="Pedestal-like QDC [mV ns]",
    ylabel="Events with QDC greater than x [%]",
    title="Pedestal-like QDC upper-tail fraction",
    yscale="log",
)
ax_pedestal_survival.grid(True, which="both", alpha=0.35)
fig_pedestal_cdf.tight_layout()
plt.show()
print(
    f"Pedestal-like events with QDC > {pedestal_qdc_tail_threshold_mV_ns:g} mV ns: "
    f"{n_pedestal_above_threshold:,}/{len(finite_pedestal_qdc):,} "
    f"({100 * pedestal_fraction_above_threshold:.3f}%)"
)
full_pedestal_above_threshold = (
    finite_pedestal_full_qdc > pedestal_qdc_tail_threshold_mV_ns
)
n_full_pedestal_above_threshold = int(np.sum(full_pedestal_above_threshold))
full_pedestal_fraction_above_threshold = (
    n_full_pedestal_above_threshold / max(len(finite_pedestal_full_qdc), 1)
)
paired_finite_pedestal_qdc = (
    np.isfinite(pedestal_qdc_values)
    & np.isfinite(pedestal_full_waveform_qdc_values)
)
original_tail_paired = (
    pedestal_qdc_values[paired_finite_pedestal_qdc]
    > pedestal_qdc_tail_threshold_mV_ns
)
full_tail_paired = (
    pedestal_full_waveform_qdc_values[paired_finite_pedestal_qdc]
    > pedestal_qdc_tail_threshold_mV_ns
)
n_original_tail_removed_by_full = int(np.sum(original_tail_paired & ~full_tail_paired))
print(
    f"Full-waveform pedestal charge > {pedestal_qdc_tail_threshold_mV_ns:g} mV ns: "
    f"{n_full_pedestal_above_threshold:,}/{len(finite_pedestal_full_qdc):,} "
    f"({100 * full_pedestal_fraction_above_threshold:.3f}%)"
)
print(
    f"Original 0-80 ns tail events falling below the threshold with "
    f"full-waveform integration: {n_original_tail_removed_by_full:,}/"
    f"{int(np.sum(original_tail_paired)):,}"
)

# Detailed view of the bounded diagnostic waveforms retained by the stream.
if pedestal_diagnostic_records:
    diagnostic_ncols = 2
    diagnostic_nrows = int(np.ceil(len(pedestal_diagnostic_records) / diagnostic_ncols))
    fig_diagnostic, diagnostic_axes = plt.subplots(
        diagnostic_nrows, diagnostic_ncols,
        figsize=(14, 4.8 * diagnostic_nrows), squeeze=False, sharex=True,
    )
    for ax, record in zip(diagnostic_axes.ravel(), pedestal_diagnostic_records):
        waveform_mV = record["waveform_mV"]
        waveform_positive_mV = -waveform_mV
        baseline_rms_mV = record["baseline_rms_mV"]
        diagnostic = record["metrics"]
        ax.plot(sample_time_ns, waveform_mV, color="tab:orange", linewidth=1.1)
        ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.6)
        ax.axhline(
            -sample_preprocessing["peak_snr_threshold"] * baseline_rms_mV,
            color="tab:orange", linestyle="--", linewidth=1.2,
            label="8-RMS height threshold",
        )
        ax.axvspan(
            *sample_baseline_window_ns, color="0.5", alpha=0.10,
            label="original cache baseline window",
        )
        ax.axvspan(
            *pedestal_qdc_window_ns, color="tab:orange", alpha=0.05, hatch="//",
            label="pedestal QDC window",
        )
        for sideband_index, sideband_ns in enumerate(
            pedestal_full_waveform_baseline_sidebands_ns
        ):
            ax.axvspan(
                *sideband_ns, color="tab:green", alpha=0.07,
                label=("full-QDC baseline sidebands" if sideband_index == 0 else None),
            )
        if diagnostic is not None:
            width_left_ns = diagnostic["width_left_ns"]
            width_right_ns = diagnostic["width_right_ns"]
            width_passes = not any(
                check.startswith("width") for check in diagnostic["failed_checks"]
            )
            ax.axvline(
                diagnostic["time_ns"], color="tab:blue", linestyle="--",
                linewidth=1.2, label="strongest candidate time",
            )
            ax.scatter(
                diagnostic["time_ns"], -diagnostic["amplitude_mV"],
                color="tab:blue", edgecolor="black", s=45, zorder=5,
            )
            ax.hlines(
                diagnostic["width_level_mV"], width_left_ns, width_right_ns,
                color=("tab:green" if width_passes else "tab:red"),
                linewidth=3.0,
            )
            diagnostic_text = (
                f"QDC(0-80 ns) = {record['qdc_mV_ns']:.3f} mV ns\n"
                f"full-waveform QDC = {record['full_waveform_qdc_mV_ns']:.3f} mV ns\n"
                f"candidate: t={diagnostic['time_ns']:.2f} ns, amplitude={diagnostic['amplitude_mV']:.3f} mV\n"
                f"full-trace SNR={diagnostic['snr']:.2f}, prominence/RMS={diagnostic['prominence_snr']:.2f}\n"
                f"FWHM={diagnostic['width_ns']:.3f} ns\n"
                f"failed: {', '.join(diagnostic['failed_checks'])}"
            )
        else:
            diagnostic_text = (
                f"QDC(0-80 ns) = {record['qdc_mV_ns']:.3f} mV ns\n"
                f"full-waveform QDC = {record['full_waveform_qdc_mV_ns']:.3f} mV ns\n"
                "No local maximum found"
            )
        ax.set_title(
            f"sample {record['population_order']}: {Path(record['event_file']).name}, "
            f"segment {record['event_segment']}", fontsize=10,
        )
        ax.text(
            0.99, 0.03, diagnostic_text, transform=ax.transAxes,
            ha="right", va="bottom", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.88),
        )
        ax.set(xlabel="Time [ns]", ylabel="Voltage [mV]")
        ax.grid(True, which="both", alpha=0.35)
    for ax in diagnostic_axes.ravel()[len(pedestal_diagnostic_records):]:
        ax.set_visible(False)
    handles, labels = diagnostic_axes.ravel()[0].get_legend_handles_labels()
    fig_diagnostic.legend(handles, labels, loc="lower center", ncols=5, fontsize=8)
    fig_diagnostic.suptitle(
        f"Pedestal-classified waveforms with QDC in "
        f"[{pedestal_qdc_diagnostic_range_mV_ns[0]:g}, "
        f"{pedestal_qdc_diagnostic_range_mV_ns[1]:g}] mV ns "
        f"(showing at most {pedestal_qdc_diagnostic_max_waveforms})",
    )
    fig_diagnostic.tight_layout(rect=(0, 0.07, 1, 0.95))
    plt.show()
else:
    print(
        f"No sampled pedestal events have QDC in "
        f"{pedestal_qdc_diagnostic_range_mV_ns} mV ns."
    )

def generic_shape_failure_text(record):
    event = sampled_events.iloc[int(record["request_order"])]
    labels = {
        "peak_width_ns": "FWHM",
        "rise_time_10_90_ns": "10–90% rise time",
        "fall_time_90_10_ns": "90–10% fall time",
    }
    failures = []
    for column, bounds in sample_generic_shape_ranges.items():
        low, high = map(float, bounds)
        value = float(event[column])
        label = labels.get(column, column)
        if not np.isfinite(value):
            failures.append(f"{label} is not finite")
        elif value < low:
            failures.append(f"{label}={value:.3g} ns < {low:g} ns minimum")
        elif value > high:
            failures.append(f"{label}={value:.3g} ns > {high:g} ns maximum")
    return "; ".join(failures) or "no failed generic bound found"

def blue_qdc_population_status(record):
    event = sampled_events.iloc[int(record["request_order"])]
    reasons = []
    if not bool(event.get("has_generic_pulse_shape", False)):
        reasons.append("failed generic shape bounds")
    elif not bool(event.get("has_pulse_shape", False)):
        reasons.append("failed learned shape bounds")
    if not bool(event.get("is_trigger_aligned", False)):
        reasons.append("outside learned trigger-time window")
    if float(event.get("snr", np.nan)) < sample_peak_snr_min:
        reasons.append(f"SNR below {sample_peak_snr_min:g}")
    if float(event.get("peak_amplitude_mV", np.nan)) > sample_peak_amplitude_max_mV:
        reasons.append("above peak-amplitude sampling cutoff")
    if not reasons and bool(event.get("signal_good", False)):
        return "included in blue QDC signal population"
    if not reasons:
        reasons.append("failed final signal_good classification")
    return "excluded from blue QDC: " + "; ".join(reasons)

def plot_signal_diagnostic_records(
    records, figure_title, color, *, peak_record_key="qualifying_peaks",
    candidate_label="accepted peak", generic_shape_failures_only=False,
    title_style="default",
):
    if not records:
        print(f"No waveforms available for: {figure_title}")
        return
    ncols = 2
    nrows = int(np.ceil(len(records) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(14, 4.8 * nrows), squeeze=False, sharex=True
    )
    for ax, record in zip(axes.ravel(), records):
        waveform = record["waveform_mV"]
        metrics = record.get("accepted_peak_metrics") or record["metrics"]
        baseline_rms = record["baseline_rms_mV"]
        ax.plot(sample_time_ns, waveform, color=color, linewidth=1.1)
        ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.6)
        ax.axhline(
            -sample_preprocessing["peak_snr_threshold"] * baseline_rms,
            color="tab:red", linestyle="--", linewidth=1.0,
            label="height threshold",
        )
        ax.axvspan(
            *sample_fixed_window_ns, color="tab:purple", alpha=0.10,
            label="fixed pulse charge window",
        )
        qualifying_peaks = record.get(peak_record_key, [])
        largest_fraction = np.nan
        if qualifying_peaks:
            strongest_peak_index = int(np.argmax([
                peak["amplitude_mV"] for peak in qualifying_peaks
            ]))
            for peak_index, peak in enumerate(qualifying_peaks):
                is_strongest = peak_index == strongest_peak_index
                peak_color = "tab:red" if is_strongest else "tab:cyan"
                first_other_index = 1 if strongest_peak_index == 0 else 0
                peak_label = (
                    f"strongest {candidate_label}" if is_strongest
                    else (f"other {candidate_label}" if peak_index == first_other_index else None)
                )
                ax.axvline(
                    peak["time_ns"], color=peak_color, linestyle="--",
                    linewidth=(1.5 if is_strongest else 1.0), alpha=0.85,
                    label=peak_label,
                )
                ax.scatter(
                    peak["time_ns"], -peak["amplitude_mV"],
                    s=(55 if is_strongest else 35), color=peak_color,
                    edgecolor="black", linewidth=0.6, zorder=6,
                )
        if metrics is not None:
            ax.hlines(
                metrics["width_level_mV"], metrics["width_left_ns"],
                metrics["width_right_ns"], color="tab:green", linewidth=3.0,
            )
            detail_parts = [
                f"QDC={record['qdc_mV_ns']:.3f} mV ns",
                f"{candidate_label}s={len(qualifying_peaks)}",
            ]
            if np.isfinite(metrics.get("snr", np.nan)):
                detail_parts.append(f"SNR={metrics['snr']:.2f}")
            if np.isfinite(metrics.get("prominence_snr", np.nan)):
                detail_parts.append(
                    f"prominence/RMS={metrics['prominence_snr']:.2f}"
                )
            if np.isfinite(metrics.get("width_ns", np.nan)):
                detail_parts.append(f"FWHM={metrics['width_ns']:.3f} ns")
            details = "; ".join(detail_parts)
            if peak_record_key == "signal_like_peaks" and len(qualifying_peaks) > 1:
                additional_peaks = [
                    peak for peak in qualifying_peaks
                    if peak["candidate_idx"] != metrics["candidate_idx"]
                ]
                if additional_peaks and metrics["amplitude_mV"] > 0:
                    largest_fraction = max(
                        peak["amplitude_mV"] for peak in additional_peaks
                    ) / metrics["amplitude_mV"]
                    details += f"; largest additional/main={largest_fraction:.3f}"
        else:
            details = f"QDC={record['qdc_mV_ns']:.3f} mV ns; no candidate"
        if generic_shape_failures_only:
            panel_title = generic_shape_failure_text(record)
        elif title_style == "multi_peak":
            cached_event = sampled_events.iloc[int(record["request_order"])]
            panel_title = (
                f"peaks={int(cached_event['n_peaks'])}; "
                f"QDC={record['qdc_mV_ns']:.3f} mV ns"
            )
        elif title_style == "candidate_dominance":
            panel_title = f"pre-candidates={len(qualifying_peaks)}"
            if np.isfinite(largest_fraction):
                panel_title += f"; largest additional/main={largest_fraction:.3f}"
        elif title_style == "candidate_dominance_final":
            panel_title = f"pre-candidates={len(qualifying_peaks)}"
            if np.isfinite(largest_fraction):
                panel_title += f"; largest additional/main={largest_fraction:.3f}"
            panel_title += "\n" + blue_qdc_population_status(record)
        elif title_style == "no_segment":
            panel_title = details
        else:
            panel_title = f"segment {record['event_segment']} — {details}"
        ax.set_title(panel_title, fontsize=9)
        ax.set(xlabel="Time [ns]", ylabel="Voltage [mV]")
        ax.grid(True, which="both", alpha=0.35)
    for ax in axes.ravel()[len(records):]:
        ax.set_visible(False)
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncols=3, fontsize=8)
    fig.suptitle(figure_title)
    fig.tight_layout(rect=(0, 0.05, 1, 0.96))
    plt.show()

plot_signal_diagnostic_records(
    multi_peak_plot_records,
    f"Cached multi-peak events (showing {len(multi_peak_plot_records)} of "
    f"{n_multi_peak_events:,})",
    "tab:green",
    title_style="multi_peak",
)
plot_signal_diagnostic_records(
    oscillatory_plot_records,
    f"Rejected oscillatory single-width-peak events (showing "
    f"{len(oscillatory_plot_records)} of {n_oscillatory_events:,})",
    "tab:red",
    peak_record_key="signal_like_peaks",
    candidate_label="pre-width qualifying candidate",
    title_style="candidate_dominance",
)
plot_signal_diagnostic_records(
    small_additional_peak_blue_plot_records,
    f"Dominant signals with small additional candidates included in the blue "
    f"QDC population (showing {len(small_additional_peak_blue_plot_records)} of "
    f"{n_small_additional_peak_blue_events:,})",
    "tab:blue",
    peak_record_key="signal_like_peaks",
    candidate_label="pre-width qualifying candidate",
    title_style="candidate_dominance_final",
)
plot_signal_diagnostic_records(
    small_additional_peak_plot_records,
    f"Accepted dominant signals with small additional candidates (showing "
    f"{len(small_additional_peak_plot_records)} of "
    f"{n_small_additional_peak_events:,})",
    "tab:olive",
    peak_record_key="signal_like_peaks",
    candidate_label="pre-width qualifying candidate",
    title_style="candidate_dominance_final",
)
plot_signal_diagnostic_records(
    generic_bad_shape_plot_records,
    f"Rejected isolated peaks outside generic shape bounds (showing "
    f"{len(generic_bad_shape_plot_records)} of {n_generic_bad_shape_events:,})",
    "tab:brown",
    generic_shape_failures_only=True,
)
plot_signal_diagnostic_records(
    peak_low_qdc_diagnostic_records,
    f"Classified signals with QDC in {peak_low_qdc_diagnostic_range_mV_ns} mV ns "
    f"(showing at most {peak_low_qdc_diagnostic_max_waveforms})",
    "tab:blue",
    title_style="no_segment",
)
plot_signal_diagnostic_records(
    wide_peak_diagnostic_records,
    f"Classified signals with FWHM > {wide_peak_diagnostic_min_width_ns:g} ns "
    f"(showing at most {wide_peak_diagnostic_max_waveforms})",
    "tab:purple",
)

print(
    f"Cached n_peaks > 1 events: {n_multi_peak_events:,}/{n_cached_events:,} "
    f"({100 * n_multi_peak_events / n_cached_events:.4f}%)"
)
print(
    f"Rejected oscillatory single-width-peak events: {n_oscillatory_events:,}/"
    f"{n_cached_events:,} ({100 * n_oscillatory_events / n_cached_events:.4f}%)"
)
print(
    f"Accepted dominant pulses with small additional candidates: "
    f"{n_small_additional_peak_events:,}/{n_cached_events:,} "
    f"({100 * n_small_additional_peak_events / n_cached_events:.4f}%)"
)
print(
    f"Rejected isolated peaks outside generic shape bounds: "
    f"{n_generic_bad_shape_events:,}/{n_cached_events:,} "
    f"({100 * n_generic_bad_shape_events / n_cached_events:.4f}%)"
)
if signal_like_bad_shape_count is None:
    print("signal_like_bad_shape is unavailable in this old cache; rerun Selection.")
else:
    print(
        f"Signal-like bad-shape events: {signal_like_bad_shape_count:,}/"
        f"{n_cached_events:,} ({100 * signal_like_bad_shape_count / n_cached_events:.4f}%)"
    )
print(
    f"Isolated good-shape + trigger-aligned signals: "
    f"{n_cached_signal_good_events:,}; "
    f"noise-like single peaks: {n_signal_noise_like_events:,}; "
    f"good-shape off-time singles: {n_signal_off_time_events:,}"
)
print(
    f"Classified signals with FWHM > {wide_peak_diagnostic_min_width_ns:g} ns: "
    f"{n_wide_peak_events:,}/{n_peak_candidate_events:,} "
    f"({100 * n_wide_peak_events / n_peak_candidate_events:.4f}% of selected peaks)"
    if n_peak_candidate_events else "No classified signal events available."
)

print(
    f"Loaded {n_pedestal_waveforms} pedestal and {n_peak_waveforms} "
    f"post-mismatch peak waveforms "
    f"with peak amplitude <= {sample_peak_amplitude_max_mV:g} mV "
    f"(raw negative pulse >= -{sample_peak_amplitude_max_mV:g} mV)"
)
print(
    f"Waveform overlay shows {len(pedestal_plot_records)} pedestal and "
    f"{len(peak_plot_records)} post-mismatch peak waveforms; QDC histograms "
    f"use all retained events ({signal_shape_survival_annotation})."
)
print(f"Fixed pulse window: {sample_fixed_window_ns} ns")
print(f"Pedestal QDC window: {pedestal_qdc_window_ns} ns")
print(
    f"Full-waveform pedestal baseline sidebands: "
    f"{pedestal_full_waveform_baseline_sidebands_ns} ns"
)
print(
    "Pedestal baseline mode: per-event mean and RMS over the entire waveform "
    "(applied after cached n_peaks == 0 classification)"
)
if len(pedestal_qdc_values):
    print(f"Pedestal QDC: mean={pedestal_qdc_values.mean():.6g}, median={np.median(pedestal_qdc_values):.6g} mV ns")
else:
    print("Pedestal QDC: unavailable (no pedestal-classified events)")
if len(peak_qdc_values):
    print(
        f"Post-mismatch peak QDC: mean={peak_qdc_values.mean():.6g}, "
        f"median={np.median(peak_qdc_values):.6g} mV ns; "
        f"{signal_shape_survival_annotation}."
    )
else:
    print("Peak QDC unavailable: no signals survived the mismatch cut.")
large_pedestal_candidates = [
    record for record in pedestal_plot_records
    if record["metrics"] is not None
    and record["metrics"]["amplitude_mV"] > 2.0
]
print(
    f"\nLarge excursions (>2 mV) among plotted cached n_peaks=0 events: "
    f"{len(large_pedestal_candidates)}"
)
for record in large_pedestal_candidates:
    diagnostic = record["metrics"]
    print(
        f"  sample {record['population_order']}: "
        f"{Path(record['event_file']).name} segment {record['event_segment']}, "
        f"t={diagnostic['time_ns']:.3f} ns, amplitude={diagnostic['amplitude_mV']:.3f} mV, "
        f"SNR={diagnostic['snr']:.2f}, prominence/RMS={diagnostic['prominence_snr']:.2f}, "
        f"width={diagnostic['width_ns']:.3f} ns: "
        f"failed {', '.join(diagnostic['failed_checks'])}"
    )

# Shape metrics were calculated during the QDC stream above. Keep their
# figures in the separate diagnostics PDF without rereading the dataset.
if shape_quality_results is None:
    print("Skipping waveform-shape diagnostics PDF: no signal template.")
else:
    signal_template_mismatch = shape_quality_results[
        "signal_template_mismatch"
    ]
    pedestal_noise_growth_ratio = shape_quality_results[
        "pedestal_noise_growth_ratio"
    ]
    pedestal_local_rms_relative_span = shape_quality_results[
        "pedestal_local_rms_relative_span"
    ]
    shape_cache_signal_events = (
        shape_cache_signal_events_before_mismatch_cut.copy()
    )
    shape_cache_signal_events = shape_cache_signal_events.rename(columns={
        current_qdc_method_fixed_column: "fixed_qdc_mV_ns"
    })
    shape_cache_signal_events["event_specific_qdc_mV_ns"] = (
        peak_qdc_values_before_mismatch_cut
    )
    shape_cache_signal_events["raw_full_trace_qdc_mV_ns"] = (
        peak_raw_full_waveform_qdc_values_before_mismatch_cut
    )
    shape_cache_signal_events["event_file"] = shape_cache_signal_events[
        "event_file"
    ].map(lambda path: str(source_by_name[Path(path).name]))
    shape_cache_signal_events["event_file"] = shape_cache_signal_events[
        "event_file"
    ].astype("category")
    shape_cache_pedestal_events = sampled_pedestals.loc[:, [
        "event_file", "event_segment", current_qdc_method_fixed_column
    ]].copy()
    shape_cache_pedestal_events = shape_cache_pedestal_events.rename(columns={
        current_qdc_method_fixed_column: "fixed_qdc_mV_ns"
    })
    shape_cache_pedestal_events["event_specific_qdc_mV_ns"] = (
        np.asarray(pedestal_qdc_values, dtype=float)
    )
    shape_cache_pedestal_events["raw_full_trace_qdc_mV_ns"] = (
        np.asarray(pedestal_raw_full_waveform_qdc_values, dtype=float)
    )
    shape_cache_pedestal_events["event_file"] = shape_cache_pedestal_events[
        "event_file"
    ].map(lambda path: str(source_by_name[Path(path).name]))
    shape_cache_pedestal_events["event_file"] = shape_cache_pedestal_events[
        "event_file"
    ].astype("category")
    save_shape_diagnostics_cache(
        shape_metrics_cache_path,
        {
            "acquisition": sample_acquisition,
            "source_cache_path": str(sample_cache_path),
            "source_cache_mtime_ns": sample_cache_path.stat().st_mtime_ns,
            "preprocessing": {
                "channel": sample_preprocessing["channel"],
                "baseline_window_ns": sample_baseline_window_ns,
            },
            "baseline_reference_time_ns": reference_time_ns,
            "baseline_reference_mV": reference_mV,
            "sample_time_ns": sample_time_ns,
            "fixed_window_ns": sample_fixed_window_ns,
            "template_relative_time_ns": shape_template_relative_time_ns,
            "signal_shape_template": signal_shape_template,
            "n_template_references": n_shape_template_references,
            "signal_template_mismatch": signal_template_mismatch,
            "pedestal_noise_growth_ratio": pedestal_noise_growth_ratio,
            "pedestal_local_rms_relative_span": (
                pedestal_local_rms_relative_span
            ),
            "signal_qdc_mV_ns": peak_qdc_values_before_mismatch_cut,
            "pedestal_qdc_mV_ns": pedestal_qdc_values,
            "signal_events": shape_cache_signal_events,
            "pedestal_events": shape_cache_pedestal_events,
            "pedestal_time_blocks": shape_pedestal_time_blocks,
            "pedestal_local_rms_window_ns": (
                shape_pedestal_local_rms_window_ns
            ),
            "qdc_live_time_s": qdc_live_time_s,
            "pedestal_event_weight_per_s": (
                qdc_pedestal_event_weight_per_s
            ),
            "signal_event_weight_per_s": qdc_signal_event_weight_per_s,
        },
    )
    print(f"Saved reduced shape-metrics cache to {shape_metrics_cache_path}")
    if not 0.0 <= shape_signal_reject_upper_fraction < 1.0:
        raise ValueError(
            "shape_signal_reject_upper_fraction must be in [0, 1)"
        )
    if shape_pedestal_growth_ratio_max <= 0.0:
        raise ValueError("shape_pedestal_growth_ratio_max must be positive")
    finite_signal_shape = np.isfinite(signal_template_mismatch)
    if np.any(finite_signal_shape):
        signal_mismatch_threshold = float(np.quantile(
            signal_template_mismatch[finite_signal_shape],
            1.0 - shape_signal_reject_upper_fraction,
        ))
    else:
        signal_mismatch_threshold = np.nan
    signal_shape_keep = (
        finite_signal_shape
        & (signal_template_mismatch <= signal_mismatch_threshold)
    )
    finite_pedestal_growth = np.isfinite(pedestal_noise_growth_ratio)
    pedestal_growth_keep = (
        finite_pedestal_growth
        & (pedestal_noise_growth_ratio <= shape_pedestal_growth_ratio_max)
    )
    finite_pedestal_rms_variation = np.isfinite(
        pedestal_local_rms_relative_span
    )
    pedestal_rms_constancy_keep = (
        finite_pedestal_rms_variation
        & (
            pedestal_local_rms_relative_span
            <= shape_pedestal_rms_variation_max
        )
    )
    pedestal_shape_keep = (
        pedestal_growth_keep & pedestal_rms_constancy_keep
    )
    pedestal_rms_constancy_percent = (
        100.0 * np.mean(pedestal_rms_constancy_keep)
        if len(pedestal_rms_constancy_keep) else np.nan
    )
    print(
        f"Signal shape cut: rejected {np.sum(~signal_shape_keep):,}/"
        f"{len(signal_shape_keep):,} at mismatch > "
        f"{signal_mismatch_threshold:.6g} "
        f"(requested upper {100 * shape_signal_reject_upper_fraction:g}%)."
    )
    print(
        f"Pedestal growth cut: kept {np.sum(pedestal_growth_keep):,}/"
        f"{len(pedestal_growth_keep):,} at growth ratio <= "
        f"{shape_pedestal_growth_ratio_max:g}."
    )
    print(
        f"Pedestal local-RMS constancy: kept "
        f"{np.sum(pedestal_rms_constancy_keep):,}/"
        f"{len(pedestal_rms_constancy_keep):,} "
        f"({pedestal_rms_constancy_percent:.2f}%) with "
        f"{shape_pedestal_local_rms_window_ns:g} ns RMS span <= "
        f"{shape_pedestal_rms_variation_max:.1%}."
    )
    print(
        f"Combined pedestal shape cuts: kept "
        f"{np.sum(pedestal_shape_keep):,}/"
        f"{len(pedestal_shape_keep):,}."
    )
    shape_diagnostics_pdf_path.parent.mkdir(parents=True, exist_ok=True)
    shape_pdf_pages = 0
    with PdfPages(shape_diagnostics_pdf_path) as shape_pdf:
        def save_shape_figure(figure):
            shape_pdf.savefig(figure, bbox_inches="tight")
            plt.close(figure)

        finite_mismatch = np.isfinite(signal_template_mismatch)
        paired_mismatch = finite_mismatch & np.isfinite(
            peak_qdc_values_before_mismatch_cut
        )
        fig_signal_shape, signal_shape_axes_grid = plt.subplots(
            2, 2, figsize=(16, 11)
        )
        signal_shape_axes = signal_shape_axes_grid.ravel()
        signal_shape_axes[0].plot(
            shape_template_relative_time_ns, signal_shape_template,
            color="black", linewidth=2.0,
        )
        signal_shape_axes[0].set(
            xlabel="Time relative to peak [ns]",
            ylabel="Normalized pulse amplitude",
            title=f"Median high-SNR signal template (N={n_shape_template_references})",
        )
        if np.any(finite_mismatch):
            signal_shape_axes[1].hist(
                signal_template_mismatch[finite_mismatch],
                bins=shape_metric_histogram_bins, histtype="step",
                linewidth=2.0, color="tab:blue",
            )
            signal_shape_axes[1].axvline(
                signal_mismatch_threshold, color="tab:red", linestyle="--",
                label=f"reject above {signal_mismatch_threshold:.4g}",
            )
        signal_shape_axes[1].set(
            xlabel="Normalized template mismatch", ylabel="Blue events",
            title="Blue signal template-mismatch distribution",
        )
        signal_shape_axes[1].legend(fontsize=8)
        if np.any(finite_mismatch):
            sorted_mismatch = np.sort(
                signal_template_mismatch[finite_mismatch]
            )
            mismatch_cdf_percent = (
                100.0 * np.arange(1, len(sorted_mismatch) + 1)
                / len(sorted_mismatch)
            )
            mismatch_cut_kept_percent = (
                100.0 * np.mean(sorted_mismatch <= signal_mismatch_threshold)
            )
            signal_shape_axes[2].step(
                sorted_mismatch, mismatch_cdf_percent, where="post",
                color="tab:blue", linewidth=1.8,
            )
            signal_shape_axes[2].axvline(
                signal_mismatch_threshold, color="tab:red", linestyle="--",
                label=(
                    f"cut={signal_mismatch_threshold:.4g}: "
                    f"{mismatch_cut_kept_percent:.2f}% kept, "
                    f"{100.0 - mismatch_cut_kept_percent:.2f}% rejected"
                ),
            )
            signal_shape_axes[2].axhline(
                mismatch_cut_kept_percent, color="tab:red",
                linestyle=":", alpha=0.7,
            )
        signal_shape_axes[2].set(
            xlabel="Normalized template mismatch",
            ylabel="Events with mismatch ≤ x [%]",
            title="Signal mismatch empirical CDF", ylim=(0.0, 100.0),
        )
        signal_shape_axes[2].legend(fontsize=8)
        if np.any(paired_mismatch):
            mismatch_qdc_hist = signal_shape_axes[3].hist2d(
                signal_template_mismatch[paired_mismatch],
                peak_qdc_values_before_mismatch_cut[paired_mismatch],
                bins=shape_metric_qdc_bins, cmap="jet", norm=LogNorm(),
            )
            fig_signal_shape.colorbar(
                mismatch_qdc_hist[3], ax=signal_shape_axes[3],
                label="Waveforms per bin",
            )
        signal_shape_axes[3].set(
            xlabel="Normalized template mismatch",
            ylabel="Event-by-event QDC [mV ns]",
            title="Blue signal mismatch vs QDC",
        )
        for ax in signal_shape_axes:
            ax.grid(True, which="both", alpha=0.3)
        fig_signal_shape.tight_layout()
        save_shape_figure(fig_signal_shape)
        shape_pdf_pages += 1

        worst_signal_records = [
            record for record in shape_quality_results["worst_signal_records"]
            if record["metric"] > signal_mismatch_threshold
        ]
        if worst_signal_records:
            ncols = 2
            nrows = int(np.ceil(len(worst_signal_records) / ncols))
            fig_worst_signal, axes = plt.subplots(
                nrows, ncols, figsize=(14, 4.2 * nrows), squeeze=False,
                sharex=True, sharey=True,
            )
            for ax, record in zip(axes.ravel(), worst_signal_records):
                ax.plot(
                    shape_template_relative_time_ns, signal_shape_template,
                    color="black", linewidth=1.8, label="median template",
                )
                ax.plot(
                    shape_template_relative_time_ns, record["normalized_shape"],
                    color="tab:blue", linewidth=1.0, label="waveform",
                )
                ax.set_title(
                    f"mismatch={record['metric']:.3g}; "
                    f"QDC={record['qdc_mV_ns']:.3g} mV ns", fontsize=9,
                )
                ax.set(xlabel="Time relative to peak [ns]", ylabel="Normalized amplitude")
                ax.grid(True, alpha=0.3)
            for ax in axes.ravel()[len(worst_signal_records):]:
                ax.set_visible(False)
            handles, labels = axes.ravel()[0].get_legend_handles_labels()
            fig_worst_signal.legend(handles, labels, loc="lower center")
            fig_worst_signal.suptitle(
                f"Rejected blue signals: top "
                f"{100 * shape_signal_reject_upper_fraction:g}% template mismatch "
                f"(threshold {signal_mismatch_threshold:.4g})"
            )
            fig_worst_signal.tight_layout(rect=(0, 0.04, 1, 0.96))
            save_shape_figure(fig_worst_signal)
            shape_pdf_pages += 1

        highest_kept_signal_records = sorted(
            [
                record
                for record in shape_quality_results["signal_boundary_candidates"]
                if record["metric"] <= signal_mismatch_threshold
            ],
            key=lambda record: record["metric"], reverse=True,
        )[:shape_worst_waveforms_to_plot]
        if highest_kept_signal_records:
            ncols = 2
            nrows = int(np.ceil(len(highest_kept_signal_records) / ncols))
            fig_highest_kept_signal, axes = plt.subplots(
                nrows, ncols, figsize=(14, 4.2 * nrows), squeeze=False,
                sharex=True, sharey=True,
            )
            for ax, record in zip(axes.ravel(), highest_kept_signal_records):
                ax.plot(
                    shape_template_relative_time_ns, signal_shape_template,
                    color="black", linewidth=1.8, label="median template",
                )
                ax.plot(
                    shape_template_relative_time_ns, record["normalized_shape"],
                    color="tab:cyan", linewidth=1.0, label="waveform",
                )
                ax.set_title(
                    f"mismatch={record['metric']:.3g}; "
                    f"QDC={record['qdc_mV_ns']:.3g} mV ns", fontsize=9,
                )
                ax.set(
                    xlabel="Time relative to peak [ns]",
                    ylabel="Normalized amplitude",
                )
                ax.grid(True, alpha=0.3)
            for ax in axes.ravel()[len(highest_kept_signal_records):]:
                ax.set_visible(False)
            handles, labels = axes.ravel()[0].get_legend_handles_labels()
            fig_highest_kept_signal.legend(
                handles, labels, loc="lower center"
            )
            fig_highest_kept_signal.suptitle(
                f"Highest-mismatch blue signals still accepted after removing "
                f"the top {100 * shape_signal_reject_upper_fraction:g}% "
                f"(threshold {signal_mismatch_threshold:.4g})"
            )
            fig_highest_kept_signal.tight_layout(rect=(0, 0.04, 1, 0.96))
            save_shape_figure(fig_highest_kept_signal)
            shape_pdf_pages += 1

        finite_growth = np.isfinite(pedestal_noise_growth_ratio)
        paired_growth = finite_growth & np.isfinite(pedestal_qdc_values)
        if np.any(finite_growth):
            fig_pedestal_shape, pedestal_shape_axes = plt.subplots(1, 2, figsize=(15, 6))
            pedestal_shape_axes[0].hist(
                pedestal_noise_growth_ratio[finite_growth],
                bins=shape_metric_histogram_bins, histtype="step",
                linewidth=2.0, color="tab:orange",
            )
            pedestal_shape_axes[0].axvline(1.0, color="black", linestyle="--")
            pedestal_shape_axes[0].axvline(
                shape_pedestal_growth_ratio_max, color="tab:red",
                linestyle="--",
                label=f"reject above {shape_pedestal_growth_ratio_max:g}",
            )
            pedestal_shape_axes[0].set(
                xlabel="Late/early robust-RMS growth ratio",
                ylabel="Pedestal events",
                title="Pedestal noise-growth distribution",
            )
            pedestal_shape_axes[0].legend(fontsize=8)
            if np.any(paired_growth):
                growth_qdc_hist = pedestal_shape_axes[1].hist2d(
                    pedestal_noise_growth_ratio[paired_growth],
                    pedestal_qdc_values[paired_growth],
                    bins=shape_metric_qdc_bins, cmap="jet", norm=LogNorm(),
                )
                fig_pedestal_shape.colorbar(
                    growth_qdc_hist[3], ax=pedestal_shape_axes[1],
                    label="Waveforms per bin",
                )
            pedestal_shape_axes[1].set(
                xlabel="Late/early robust-RMS growth ratio",
                ylabel="Pedestal 0–80 ns QDC [mV ns]",
                title="Pedestal noise growth vs QDC",
            )
            for ax in pedestal_shape_axes:
                ax.grid(True, which="both", alpha=0.3)
            fig_pedestal_shape.tight_layout()
            save_shape_figure(fig_pedestal_shape)
            shape_pdf_pages += 1

        finite_rms_variation = np.isfinite(
            pedestal_local_rms_relative_span
        )
        paired_rms_variation = (
            finite_rms_variation & np.isfinite(pedestal_qdc_values)
        )
        if np.any(finite_rms_variation):
            finite_rms_values = pedestal_local_rms_relative_span[
                finite_rms_variation
            ]
            rms_constancy_survival_percent = 100.0 * np.mean(
                finite_rms_values <= shape_pedestal_rms_variation_max
            )
            sorted_rms_values = np.sort(finite_rms_values)
            rms_cdf_percent = (
                100.0 * np.arange(1, len(sorted_rms_values) + 1)
                / len(sorted_rms_values)
            )
            fig_rms_constancy, rms_constancy_axes = plt.subplots(
                1, 3, figsize=(19, 6)
            )
            rms_constancy_axes[0].hist(
                finite_rms_values, bins=shape_metric_histogram_bins,
                histtype="step", linewidth=2.0, color="tab:brown",
            )
            rms_constancy_axes[0].axvline(
                shape_pedestal_rms_variation_max, color="tab:red",
                linestyle="--",
                label=(
                    f"cut={shape_pedestal_rms_variation_max:.1%}; "
                    f"survival={rms_constancy_survival_percent:.2f}%"
                ),
            )
            rms_constancy_axes[0].set(
                xlabel="Local RMS relative span (P90−P10)/median",
                ylabel="Pedestal events",
                title="Pedestal local-RMS constancy",
            )
            rms_constancy_axes[0].legend(fontsize=8)
            rms_constancy_axes[1].step(
                sorted_rms_values, rms_cdf_percent, where="post",
                color="tab:brown", linewidth=1.8,
            )
            rms_constancy_axes[1].axvline(
                shape_pedestal_rms_variation_max, color="tab:red",
                linestyle="--",
                label=f"{rms_constancy_survival_percent:.2f}% retained",
            )
            rms_constancy_axes[1].axhline(
                rms_constancy_survival_percent, color="tab:red",
                linestyle=":", alpha=0.7,
            )
            rms_constancy_axes[1].set(
                xlabel="Local RMS relative span (P90−P10)/median",
                ylabel="Events with span ≤ x [%]",
                title="Pedestal local-RMS span CDF", ylim=(0, 100),
            )
            rms_constancy_axes[1].legend(fontsize=8)
            if np.any(paired_rms_variation):
                rms_qdc_hist = rms_constancy_axes[2].hist2d(
                    pedestal_local_rms_relative_span[paired_rms_variation],
                    pedestal_qdc_values[paired_rms_variation],
                    bins=shape_metric_qdc_bins, cmap="jet", norm=LogNorm(),
                )
                fig_rms_constancy.colorbar(
                    rms_qdc_hist[3], ax=rms_constancy_axes[2],
                    label="Waveforms per bin",
                )
            rms_constancy_axes[2].set(
                xlabel="Local RMS relative span (P90−P10)/median",
                ylabel="Pedestal 0–80 ns QDC [mV ns]",
                title="Pedestal local-RMS variation vs QDC",
            )
            for ax in rms_constancy_axes:
                ax.grid(True, which="both", alpha=0.3)
            fig_rms_constancy.suptitle(
                f"Robust RMS in {shape_pedestal_local_rms_window_ns:g} ns "
                "windows across the full trace"
            )
            fig_rms_constancy.tight_layout(rect=(0, 0, 1, 0.94))
            save_shape_figure(fig_rms_constancy)
            shape_pdf_pages += 1

        before_pedestal_qdc = pedestal_qdc_values[
            np.isfinite(pedestal_qdc_values)
        ]
        before_signal_qdc = peak_qdc_values_before_mismatch_cut[
            np.isfinite(peak_qdc_values_before_mismatch_cut)
        ]
        before_combined_qdc = np.concatenate([
            before_pedestal_qdc, before_signal_qdc
        ])
        kept_pedestal_qdc = pedestal_qdc_values[
            pedestal_shape_keep & np.isfinite(pedestal_qdc_values)
        ]
        kept_signal_qdc = peak_qdc_values_before_mismatch_cut[
            signal_shape_keep
            & np.isfinite(peak_qdc_values_before_mismatch_cut)
        ]
        kept_combined_qdc = np.concatenate([
            kept_pedestal_qdc, kept_signal_qdc
        ])
        if len(before_combined_qdc):
            def shape_survival_percent(count, total):
                return 100.0 * count / total if total else np.nan

            final_pedestal_survival = shape_survival_percent(
                len(kept_pedestal_qdc), len(before_pedestal_qdc)
            )
            final_signal_survival = shape_survival_percent(
                len(kept_signal_qdc), len(before_signal_qdc)
            )
            shape_qdc_low = float(np.min(before_combined_qdc))
            shape_qdc_high = float(np.max(before_combined_qdc))
            if shape_qdc_low == shape_qdc_high:
                shape_qdc_low -= 0.5
                shape_qdc_high += 0.5
            shape_qdc_edges = np.linspace(
                shape_qdc_low, shape_qdc_high, qdc_histogram_bins + 1
            )
            fig_post_shape_qdc, post_shape_axes = plt.subplots(
                4, 2, figsize=(16, 20), sharex=True, sharey=True
            )
            shape_qdc_states = (
                ("Before either cut", before_pedestal_qdc, before_signal_qdc),
                (
                    "After mismatch cut only", before_pedestal_qdc,
                    kept_signal_qdc,
                ),
                (
                    "After pedestal cuts only", kept_pedestal_qdc,
                    before_signal_qdc,
                ),
                ("After both cuts", kept_pedestal_qdc, kept_signal_qdc),
            )
            for row, (state, pedestal_values, signal_values) in enumerate(
                shape_qdc_states
            ):
                separate_ax = post_shape_axes[row, 0]
                combined_ax = post_shape_axes[row, 1]
                pedestal_survival = shape_survival_percent(
                    len(pedestal_values), len(before_pedestal_qdc)
                )
                signal_survival = shape_survival_percent(
                    len(signal_values), len(before_signal_qdc)
                )
                separate_ax.hist(
                    pedestal_values, bins=shape_qdc_edges, histtype="step",
                    linewidth=2.0, color="tab:orange", alpha=0.7,
                    weights=qdc_histogram_weights(
                        pedestal_values, "pedestal"
                    ),
                    label=(
                        f"pedestal (N={len(pedestal_values):,}; "
                        f"survival={pedestal_survival:.2f}%)"
                    ),
                )
                separate_ax.hist(
                    signal_values, bins=shape_qdc_edges, histtype="step",
                    linewidth=2.0, color="tab:blue", alpha=0.7,
                    weights=qdc_histogram_weights(signal_values, "signal"),
                    label=(
                        f"signal (N={len(signal_values):,}; "
                        f"survival={signal_survival:.2f}%)"
                    ),
                )
                separate_ax.set_title(
                    f"{state} — separate populations"
                )
                combined_values = np.concatenate([
                    pedestal_values, signal_values
                ])
                combined_survival = shape_survival_percent(
                    len(combined_values), len(before_combined_qdc)
                )
                combined_ax.hist(
                    combined_values, bins=shape_qdc_edges, histtype="step",
                    linewidth=2.2, color="tab:purple", alpha=0.7,
                    weights=combined_qdc_histogram_weights(
                        pedestal_values, signal_values
                    ),
                    label=(
                        f"combined (N={len(combined_values):,}; "
                        f"survival={combined_survival:.2f}%)"
                    ),
                )
                combined_ax.set_title(
                    f"{state} — combined population"
                )
            for ax in post_shape_axes.ravel():
                ax.set(
                    xlabel="Event-by-event QDC [mV ns]",
                    ylabel=qdc_histogram_y_label, yscale="log",
                )
                ax.grid(True, which="both", alpha=0.3)
                ax.legend(fontsize=8)
            fig_post_shape_qdc.suptitle(
                f"QDC before/after shape cuts: reject top "
                f"{100 * shape_signal_reject_upper_fraction:g}% signal mismatch; "
                f"pedestal growth > {shape_pedestal_growth_ratio_max:g} or "
                f"local RMS span > {shape_pedestal_rms_variation_max:.1%}\n"
                f"Final cut survival — signal: {final_signal_survival:.2f}%; "
                f"pedestal: {final_pedestal_survival:.2f}%"
            )
            fig_post_shape_qdc.tight_layout(rect=(0, 0, 1, 0.94))
            save_shape_figure(fig_post_shape_qdc)
            shape_pdf_pages += 1

        worst_pedestal_records = [
            record for record in shape_quality_results["worst_pedestal_records"]
            if record["metric"] > shape_pedestal_growth_ratio_max
        ]
        if worst_pedestal_records:
            ncols = 2
            nrows = int(np.ceil(len(worst_pedestal_records) / ncols))
            fig_worst_pedestal, axes = plt.subplots(
                nrows, ncols, figsize=(14, 4.2 * nrows), squeeze=False, sharex=True
            )
            block_edges_ns = np.linspace(
                sample_time_ns[0], sample_time_ns[-1], shape_pedestal_time_blocks + 1
            )
            for ax, record in zip(axes.ravel(), worst_pedestal_records):
                ax.plot(sample_time_ns, record["waveform_mV"], color="tab:orange", linewidth=1.0)
                for edge_ns in block_edges_ns[1:-1]:
                    ax.axvline(edge_ns, color="0.5", linestyle=":", alpha=0.6)
                ax.set_title(
                    f"growth={record['metric']:.3g}; "
                    f"QDC={record['qdc_mV_ns']:.3g} mV ns", fontsize=9,
                )
                ax.set(xlabel="Time [ns]", ylabel="Median-centered voltage [mV]")
                ax.grid(True, alpha=0.3)
            for ax in axes.ravel()[len(worst_pedestal_records):]:
                ax.set_visible(False)
            fig_worst_pedestal.suptitle(
                f"Rejected pedestals: noise-growth ratio > "
                f"{shape_pedestal_growth_ratio_max:g}"
            )
            fig_worst_pedestal.tight_layout(rect=(0, 0, 1, 0.96))
            save_shape_figure(fig_worst_pedestal)
            shape_pdf_pages += 1

        highest_kept_pedestal_records = shape_quality_results[
            "accepted_pedestal_records"
        ][:shape_worst_waveforms_to_plot]
        if highest_kept_pedestal_records:
            ncols = 2
            nrows = int(np.ceil(len(highest_kept_pedestal_records) / ncols))
            fig_highest_kept_pedestal, axes = plt.subplots(
                nrows, ncols, figsize=(14, 4.2 * nrows), squeeze=False,
                sharex=True,
            )
            block_edges_ns = np.linspace(
                sample_time_ns[0], sample_time_ns[-1],
                shape_pedestal_time_blocks + 1,
            )
            for ax, record in zip(
                axes.ravel(), highest_kept_pedestal_records
            ):
                ax.plot(
                    sample_time_ns, record["waveform_mV"],
                    color="tab:green", linewidth=1.0,
                )
                for edge_ns in block_edges_ns[1:-1]:
                    ax.axvline(
                        edge_ns, color="0.5", linestyle=":", alpha=0.6
                    )
                ax.set_title(
                    f"growth={record['metric']:.3g}; "
                    f"QDC={record['qdc_mV_ns']:.3g} mV ns", fontsize=9,
                )
                ax.set(
                    xlabel="Time [ns]",
                    ylabel="Median-centered voltage [mV]",
                )
                ax.grid(True, alpha=0.3)
            for ax in axes.ravel()[len(highest_kept_pedestal_records):]:
                ax.set_visible(False)
            fig_highest_kept_pedestal.suptitle(
                f"Highest-growth pedestals still accepted below ratio "
                f"{shape_pedestal_growth_ratio_max:g}"
            )
            fig_highest_kept_pedestal.tight_layout(rect=(0, 0.04, 1, 0.96))
            save_shape_figure(fig_highest_kept_pedestal)
            shape_pdf_pages += 1

        def plot_pedestal_rms_span_records(records, title, color):
            if not records:
                return 0
            ncols = 2
            nrows = int(np.ceil(len(records) / ncols))
            figure, axes = plt.subplots(
                nrows, ncols, figsize=(14, 4.2 * nrows), squeeze=False,
                sharex=True,
            )
            local_edges_ns = np.arange(
                sample_time_ns[0],
                sample_time_ns[-1] + 0.5 * shape_pedestal_local_rms_window_ns,
                shape_pedestal_local_rms_window_ns,
            )
            for ax, record in zip(axes.ravel(), records):
                ax.plot(
                    sample_time_ns, record["waveform_mV"],
                    color=color, linewidth=1.0,
                )
                for edge_ns in local_edges_ns[1:-1]:
                    ax.axvline(
                        edge_ns, color="0.6", linestyle=":", alpha=0.45
                    )
                ax.set_title(
                    f"local RMS span={record['metric']:.3g}; "
                    f"QDC={record['qdc_mV_ns']:.3g} mV ns", fontsize=9,
                )
                ax.set(
                    xlabel="Time [ns]",
                    ylabel="Median-centered voltage [mV]",
                )
                ax.grid(True, alpha=0.3)
            for ax in axes.ravel()[len(records):]:
                ax.set_visible(False)
            figure.suptitle(title)
            figure.tight_layout(rect=(0, 0.04, 1, 0.96))
            save_shape_figure(figure)
            return 1

        rejected_rms_span_records = [
            record for record in shape_quality_results[
                "worst_pedestal_rms_variation_records"
            ]
            if record["metric"] > shape_pedestal_rms_variation_max
        ]
        shape_pdf_pages += plot_pedestal_rms_span_records(
            rejected_rms_span_records,
            "Rejected pedestals: local RMS relative span > "
            f"{shape_pedestal_rms_variation_max:.1%}",
            "tab:red",
        )
        accepted_rms_span_records = shape_quality_results[
            "accepted_pedestal_rms_variation_records"
        ][:shape_worst_waveforms_to_plot]
        shape_pdf_pages += plot_pedestal_rms_span_records(
            accepted_rms_span_records,
            "Highest local-RMS span still accepted below "
            f"{shape_pedestal_rms_variation_max:.1%}",
            "tab:olive",
        )

    print(
        f"Saved {shape_pdf_pages}-page waveform-shape diagnostics PDF to "
        f"{shape_diagnostics_pdf_path}"
    )

if testing_summary_pdf_path is not None:
    if _testing_pdf_pages is not None:
        _testing_pdf_pages.close()
        _testing_pdf_pages = None
        _testing_pdf_temporary_path.replace(testing_summary_pdf_path)
    plt.show = _testing_original_show
    if _testing_pdf_page_count:
        print(
            f"Saved {_testing_pdf_page_count}-page testing summary PDF to "
            f"{testing_summary_pdf_path}"
        )
    else:
        print("No testing figures were produced; summary PDF was not replaced.")


In [ ]:
# OPTIONAL: run this cell only to compare QDC methods across voltages.
# It reads the reduced per-voltage caches written by the main Testing cell.
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# #################### MULTI-VOLTAGE QDC CONTROLS #################
# Select the voltage curves and set this comparison's independent paths.
voltage_qdc_comparison_voltages = (750,775,800,825, 850)
voltage_qdc_comparison_cache_dir = Path(
    "plots/dark_counts/fit_data/20_8"
)
voltage_qdc_comparison_output_dir = Path("plots/dark_counts_20_8")
voltage_qdc_comparison_acquisition_template = (
    "WA0089_{voltage}V_Dark_20microV_trig"
)
voltage_qdc_comparison_charge_column = "charge_fixed_pulse_window_mV_ns"
voltage_qdc_comparison_bins = 100
# False gives waveforms/s/bin when every voltage has timing, else counts/bin.
voltage_qdc_comparison_density = False
voltage_qdc_comparison_use_timing_rate_units = True
voltage_qdc_comparison_timing_time_column = "Acquisition_s"
voltage_qdc_comparison_histogram_alpha = 0.4
voltage_qdc_comparison_empty_edge_bins = 2  # empty bins below/above data
voltage_qdc_comparison_min_pedestal_events = 2  # per voltage curve
# (0, 1) shows the full observed pedestal QDC minimum–maximum range.
voltage_qdc_comparison_pedestal_range_quantiles = (0.0, 1.0)
# Normally the first cell creates each voltage cache during its regular pass.
voltage_qdc_comparison_build_missing_caches = False
voltage_qdc_comparison_cache_version = 5  # includes applied signal-mismatch cut
voltage_qdc_comparison_reduced_cache_dir = (
    voltage_qdc_comparison_output_dir / "voltage_qdc_method_cache"
)
voltage_qdc_comparison_pdf_path = (
    voltage_qdc_comparison_output_dir
    / "normalized_qdc_methods_voltage_comparison.pdf"
)
# #################################################################

# QDC overlays across voltage for all three integration methods.
# Raw-derived reduced arrays are cached so later Testing runs avoid rescanning.
voltage_qdc_distributions = {
    "fixed": {}, "event_specific": {}, "raw_full_trace": {}
}
voltage_qdc_population_distributions = {
    method: {"pedestal": {}, "signal_good": {}}
    for method in voltage_qdc_distributions
}
voltage_qdc_event_weights_per_s = {}
voltage_qdc_live_times_s = {}
def padded_qdc_histogram_edges(data_low, data_high):
    """Return edges and limits with empty bins outside the data range."""
    n_bins = int(voltage_qdc_comparison_bins)
    edge_bins = int(voltage_qdc_comparison_empty_edge_bins)
    if n_bins < 1 or edge_bins < 0:
        raise ValueError("QDC bin count must be positive and edge bins nonnegative")
    if data_low == data_high:
        data_low -= 0.5
        data_high += 0.5
    bin_width = (data_high - data_low) / n_bins
    plot_low = data_low - edge_bins * bin_width
    plot_high = data_high + edge_bins * bin_width
    edges = np.linspace(plot_low, plot_high, n_bins + 2 * edge_bins + 1)
    return edges, plot_low, plot_high

voltage_qdc_comparison_reduced_cache_dir.mkdir(parents=True, exist_ok=True)
for voltage in voltage_qdc_comparison_voltages:
    acquisition = voltage_qdc_comparison_acquisition_template.format(
        voltage=voltage
    )
    cache_path = Path(voltage_qdc_comparison_cache_dir) / f"{acquisition}_df.pkl"
    if not cache_path.is_file():
        print(f"Skipping {voltage} V comparison: missing {cache_path}")
        continue
    reduced_path = (
        voltage_qdc_comparison_reduced_cache_dir
        / f"{acquisition}_qdc_methods.npz"
    )
    cache_mtime_ns = cache_path.stat().st_mtime_ns
    reduced = None
    reduced_populations = None
    reduced_event_weights = None
    reduced_live_time_s = np.nan
    if reduced_path.is_file():
        with np.load(reduced_path, allow_pickle=False) as saved:
            saved_cache_version = int(saved["cache_version"].item())
            if (
                saved_cache_version == voltage_qdc_comparison_cache_version
                and int(saved["source_cache_mtime_ns"].item()) == cache_mtime_ns
            ):
                reduced = {
                    method: saved[method].copy()
                    for method in voltage_qdc_distributions
                }
                reduced_populations = {
                    method: {
                        population: saved[f"{method}_{population}"].copy()
                        for population in ("pedestal", "signal_good")
                    }
                    for method in voltage_qdc_distributions
                }
                if saved_cache_version >= 4:
                    reduced_event_weights = {
                        "pedestal": float(
                            saved["pedestal_event_weight_per_s"].item()
                        ),
                        "signal_good": float(
                            saved["signal_good_event_weight_per_s"].item()
                        ),
                    }
                    reduced_live_time_s = float(saved["live_time_s"].item())
                else:
                    reduced_event_weights = {
                        "pedestal": np.nan, "signal_good": np.nan
                    }
                print(f"Loaded {voltage:g} V reduced QDC methods from {reduced_path}")
    if reduced is None and not voltage_qdc_comparison_build_missing_caches:
        print(
            f"Skipping {voltage:g} V comparison: reduced cache is missing "
            f"or stale at {reduced_path}. Run the first Testing cell for "
            f"{acquisition} to create it."
        )
        continue
    if reduced is None:
        print(
            f"Skipping {voltage:g} V comparison: a mismatch-aware reduced "
            "cache must be created by running the first Testing cell for "
            f"{acquisition}. The comparison cell cannot learn/apply the "
            "signal template by itself."
        )
        continue
    if reduced is None:
        print(f"Computing reduced QDC methods for {voltage:g} V ...", flush=True)
        voltage_df = pd.read_pickle(cache_path)
        required_voltage_columns = {
            "n_peaks", "signal_good", voltage_qdc_comparison_charge_column
        }
        missing_voltage_columns = required_voltage_columns.difference(
            voltage_df.columns
        )
        if missing_voltage_columns:
            print(
                f"Skipping {voltage} V comparison: missing columns "
                f"{sorted(missing_voltage_columns)}"
            )
            del voltage_df
            continue
        voltage_pedestals = voltage_df.loc[voltage_df["n_peaks"].eq(0)].copy()
        voltage_signals = voltage_df.loc[voltage_df["signal_good"]].copy()
        fixed_pedestal_values = voltage_pedestals[
            voltage_qdc_comparison_charge_column
        ].to_numpy(dtype=float)
        fixed_signal_values = voltage_signals[
            voltage_qdc_comparison_charge_column
        ].to_numpy(dtype=float)
        voltage_manifest = load_selection_cache_provenance(
            voltage_df, expected_acquisition=acquisition
        )
        voltage_preprocessing = voltage_manifest["preprocessing"]
        voltage_source_by_name = resolve_cached_event_files(
            voltage_manifest["source_files"], search_roots=(Path("PMT_Data"),)
        )
        voltage_timing_exposure = (
            load_acquisition_timing_exposure(
                voltage_source_by_name.values(),
                acquisition=acquisition,
                time_column=voltage_qdc_comparison_timing_time_column,
            )
            if voltage_qdc_comparison_use_timing_rate_units else None
        )
        reduced_live_time_s = (
            np.nan if voltage_timing_exposure is None
            else voltage_timing_exposure["live_time_s"]
        )
        voltage_event_weight = (
            np.nan if not np.isfinite(reduced_live_time_s)
            else 1.0 / reduced_live_time_s
        )
        reduced_event_weights = {
            "pedestal": voltage_event_weight,
            "signal_good": voltage_event_weight,
        }
        voltage_events = pd.concat(
            [voltage_pedestals, voltage_signals], ignore_index=True
        )
        voltage_events["qdc_population"] = (
            ["pedestal"] * len(voltage_pedestals)
            + ["peak"] * len(voltage_signals)
        )
        voltage_events["event_file"] = voltage_events["event_file"].map(
            lambda path: str(voltage_source_by_name[Path(path).name])
        )
        voltage_reference_time_ns = None
        voltage_reference_mV = None
        voltage_reference_path = voltage_preprocessing.get("baseline_reference_path")
        if voltage_reference_path is not None:
            voltage_reference = load_baseline_reference(voltage_reference_path)
            voltage_reference_time_ns = voltage_reference["time_ns"]
            voltage_reference_mV = voltage_reference["baseline_template_mV"]
        voltage_streamed = stream_event_charge_method_comparison(
            voltage_events,
            channel=voltage_preprocessing["channel"],
            baseline_window_ns=tuple(voltage_preprocessing["baseline_window_ns"]),
            baseline_reference_time_ns=voltage_reference_time_ns,
            baseline_reference_mV=voltage_reference_mV,
            pedestal_qdc_window_ns=(0.0, 80.0),
            read_chunk_size=qdc_read_chunk_size,
        )
        reduced_populations = {
            "fixed": {
                "pedestal": fixed_pedestal_values,
                "signal_good": fixed_signal_values,
            },
            "event_specific": {
                "pedestal": voltage_streamed[
                    "pedestal_event_specific_qdc_mV_ns"
                ],
                "signal_good": voltage_streamed[
                    "peak_event_specific_qdc_mV_ns"
                ],
            },
            "raw_full_trace": {
                "pedestal": voltage_streamed[
                    "pedestal_raw_full_waveform_qdc_mV_ns"
                ],
                "signal_good": voltage_streamed[
                    "peak_raw_full_waveform_qdc_mV_ns"
                ],
            },
        }
        reduced_populations = {
            method: {
                population: values[np.isfinite(values)]
                for population, values in populations.items()
            }
            for method, populations in reduced_populations.items()
        }
        reduced = {
            method: np.concatenate(list(populations.values()))
            for method, populations in reduced_populations.items()
        }
        reduced_temporary_path = reduced_path.with_suffix(
            reduced_path.suffix + ".partial"
        )
        with reduced_temporary_path.open("wb") as reduced_file:
            np.savez_compressed(
                reduced_file,
                cache_version=np.asarray(voltage_qdc_comparison_cache_version),
                source_cache_mtime_ns=np.asarray(cache_mtime_ns),
                live_time_s=np.asarray(reduced_live_time_s),
                pedestal_event_weight_per_s=np.asarray(
                    reduced_event_weights["pedestal"]
                ),
                signal_good_event_weight_per_s=np.asarray(
                    reduced_event_weights["signal_good"]
                ),
                **reduced,
                **{
                    f"{method}_{population}": values
                    for method, populations in reduced_populations.items()
                    for population, values in populations.items()
                },
            )
        reduced_temporary_path.replace(reduced_path)
        del voltage_df, voltage_pedestals, voltage_signals
        del voltage_events, voltage_streamed
        gc.collect()
    voltage_qdc_event_weights_per_s[float(voltage)] = (
        reduced_event_weights
    )
    voltage_qdc_live_times_s[float(voltage)] = reduced_live_time_s
    for method, values in reduced.items():
        if len(values):
            voltage_qdc_distributions[method][float(voltage)] = values
        for population, population_values in reduced_populations[method].items():
            if len(population_values):
                voltage_qdc_population_distributions[method][population][
                    float(voltage)
                ] = population_values

if any(voltage_qdc_distributions.values()):
    method_titles = {
        "fixed": "Learned fixed window for every event",
        "event_specific": (
            "Signals: peak−FWHM to peak+2×FWHM; pedestals: 0–80 ns"
        ),
        "raw_full_trace": "Entire raw waveform; no baseline subtraction",
    }
    requested_voltages = sorted({
        voltage for distributions in voltage_qdc_distributions.values()
        for voltage in distributions
    })
    voltage_color_map = dict(zip(
        requested_voltages,
        plt.cm.jet(np.linspace(0.05, 0.95, len(requested_voltages))),
    ))
    voltage_qdc_comparison_uses_rates = (
        not voltage_qdc_comparison_density
        and all(
            voltage in voltage_qdc_event_weights_per_s
            and all(np.isfinite(list(
                voltage_qdc_event_weights_per_s[voltage].values()
            )))
            for voltage in requested_voltages
        )
    )
    if (
        voltage_qdc_comparison_use_timing_rate_units
        and not voltage_qdc_comparison_density
        and not voltage_qdc_comparison_uses_rates
    ):
        print(
            "At least one compared voltage lacks timing exposure; using "
            "event counts for every voltage to avoid mixing units."
        )

    def voltage_population_histogram_weights(voltage, values, population):
        if not voltage_qdc_comparison_uses_rates:
            return None
        return np.full(
            len(values),
            voltage_qdc_event_weights_per_s[voltage][population],
            dtype=float,
        )

    def voltage_combined_histogram_weights(voltage, method):
        if not voltage_qdc_comparison_uses_rates:
            return None
        parts = []
        for population in ("pedestal", "signal_good"):
            values = voltage_qdc_population_distributions[method][
                population
            ].get(voltage, np.asarray([], dtype=float))
            parts.append(voltage_population_histogram_weights(
                voltage, values, population
            ))
        return np.concatenate(parts)

    voltage_qdc_y_label = (
        "Normalized density"
        if voltage_qdc_comparison_density
        else (
            "Waveforms / s / bin"
            if voltage_qdc_comparison_uses_rates else "Events / bin"
        )
    )
    fig_voltage_qdc, voltage_axes = plt.subplots(1, 3, figsize=(22, 6.5))
    for ax, (method, distributions) in zip(
        voltage_axes, voltage_qdc_distributions.items()
    ):
        if not distributions:
            ax.set_visible(False)
            continue
        all_method_qdc = np.concatenate(list(distributions.values()))
        method_low = float(np.min(all_method_qdc))
        method_high = float(np.max(all_method_qdc))
        method_edges, method_plot_low, method_plot_high = (
            padded_qdc_histogram_edges(method_low, method_high)
        )
        for voltage, charge_values in sorted(distributions.items()):
            ax.hist(
                charge_values, bins=method_edges,
                density=voltage_qdc_comparison_density,
                weights=voltage_combined_histogram_weights(voltage, method),
                histtype="step", linewidth=1.8,
                alpha=voltage_qdc_comparison_histogram_alpha,
                color=voltage_color_map[voltage],
                label=f"{voltage:g} V (N={len(charge_values):,})",
            )
        ax.set(
            xlim=(method_plot_low, method_plot_high),
            xlabel="QDC [mV ns]", ylabel=voltage_qdc_y_label,
            yscale="log", title=method_titles[method],
        )
        ax.grid(True, which="both", alpha=0.35)
        ax.legend(fontsize=8)
    fig_voltage_qdc.suptitle(
        f"QDC {'rates' if voltage_qdc_comparison_uses_rates else 'counts'} "
        "across voltages — accepted pedestal + signal_good; "
        "before experimental shape cuts"
    )
    fig_voltage_qdc.tight_layout(rect=(0, 0, 1, 0.94))
    population_labels = {
        "pedestal": "Pedestal (n_peaks = 0)",
        "signal_good": "signal_good",
    }
    population_figures = {}
    for population in ("pedestal", "signal_good"):
        figure, axes = plt.subplots(1, 3, figsize=(22, 6.5))
        population_figures[population] = figure
        for ax, method in zip(axes, voltage_qdc_distributions):
            distributions = voltage_qdc_population_distributions[method][population]
            if population == "pedestal":
                distributions = {
                    voltage: values
                    for voltage, values in distributions.items()
                    if len(values) >= voltage_qdc_comparison_min_pedestal_events
                }
            if not distributions:
                ax.set_visible(False)
                continue
            all_population_qdc = np.concatenate(list(distributions.values()))
            if population == "pedestal":
                population_low, population_high = np.quantile(
                    all_population_qdc,
                    voltage_qdc_comparison_pedestal_range_quantiles,
                )
                population_low = float(population_low)
                population_high = float(population_high)
            else:
                population_low = float(np.min(all_population_qdc))
                population_high = float(np.max(all_population_qdc))
            population_edges, population_plot_low, population_plot_high = (
                padded_qdc_histogram_edges(
                    population_low, population_high
                )
            )
            for voltage, charge_values in sorted(distributions.items()):
                ax.hist(
                    charge_values, bins=population_edges,
                    density=voltage_qdc_comparison_density,
                    weights=voltage_population_histogram_weights(
                        voltage, charge_values, population
                    ),
                    histtype="step", linewidth=1.8,
                    alpha=voltage_qdc_comparison_histogram_alpha,
                    color=voltage_color_map[voltage],
                    label=f"{voltage:g} V (N={len(charge_values):,})",
                )
            ax.set(
                xlim=(population_plot_low, population_plot_high),
                xlabel="QDC [mV ns]", ylabel=voltage_qdc_y_label,
                yscale=("linear" if population == "pedestal" else "log"),
                title=method_titles[method],
            )
            ax.grid(True, which="both", alpha=0.35)
            ax.legend(fontsize=8)
        if population == "pedestal":
            quantile_low, quantile_high = (
                voltage_qdc_comparison_pedestal_range_quantiles
            )
            range_note = (
                f"; x limits show q={quantile_low:g}–{quantile_high:g} "
                "of the included pedestal samples"
                f"; curves require N≥"
                f"{voltage_qdc_comparison_min_pedestal_events}"
            )
        else:
            range_note = ""
        figure.suptitle(
            f"QDC {'rates' if voltage_qdc_comparison_uses_rates else 'counts'} "
            f"across voltages — {population_labels[population]}"
            f"{range_note}; before experimental shape cuts"
        )
        figure.tight_layout(rect=(0, 0, 1, 0.94))
    voltage_qdc_comparison_pdf_path.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(voltage_qdc_comparison_pdf_path) as voltage_pdf:
        voltage_pdf.savefig(fig_voltage_qdc, bbox_inches="tight")
        voltage_pdf.savefig(
            population_figures["pedestal"], bbox_inches="tight"
        )
        voltage_pdf.savefig(
            population_figures["signal_good"], bbox_inches="tight"
        )
    plt.close(fig_voltage_qdc)
    for figure in population_figures.values():
        plt.close(figure)
    print(
        f"Saved 3-page multi-voltage QDC "
        f"{'rate' if voltage_qdc_comparison_uses_rates else 'count'} "
        f"comparison to "
        f"{voltage_qdc_comparison_pdf_path}"
    )



In [ ]:
# OPTIONAL FAST REPLOT: uses reduced metrics from the last full run.
# It recalculates cuts/plots and reads only the few displayed waveforms.
# Fast knobs: mismatch/pedestal cut thresholds and plot counts/bins.
# Rerun the first cell after changing event sampling, QDC/baseline settings,
# template construction, pedestal time blocks, or the source Selection cache.
from pathlib import Path
import importlib
import numpy as np
import pmt.testing_summary as pmt_testing_summary
pmt_testing_summary = importlib.reload(pmt_testing_summary)

# #################### FAST SHAPE-CUT CONTROLS ###################
shape_replot_acquisition = globals().get(
    "sample_acquisition", "WA0089_775V_Dark_20microV_trig"
)
shape_replot_signal_reject_upper_fraction = 0.05
shape_replot_pedestal_growth_ratio_max = 1.3
shape_replot_pedestal_rms_variation_max = 0.20
shape_replot_worst_waveforms_to_plot = 10
shape_replot_metric_histogram_bins = 60
shape_replot_metric_qdc_bins = 100
shape_replot_qdc_histogram_bins = 40
# Select examples by their fixed-window QDC, then compare the same events
# under fixed-window, event-specific, and raw-full-trace integration.
shape_replot_fixed_qdc_range_mV_ns = (0.0, 5.0)
shape_replot_fixed_qdc_waveforms_to_plot = 6
shape_replot_fixed_qdc_population = "both"  # both, pedestal, or signal
shape_replot_output_dir = Path("plots/dark_counts_16_8")
# #################################################################

shape_replot_cache_path = (
    shape_replot_output_dir / "shape_metric_cache"
    / f"{shape_replot_acquisition}_shape_metrics.pkl"
)
shape_replot_pdf_path = (
    shape_replot_output_dir
    / f"{shape_replot_acquisition}_waveform_shape_diagnostics.pdf"
)
if not shape_replot_cache_path.is_file():
    raise FileNotFoundError(
        f"Missing {shape_replot_cache_path}; run the first Testing cell once "
        f"for {shape_replot_acquisition}."
    )
shape_replot_result = (
    pmt_testing_summary.generate_shape_diagnostics_pdf_from_cache(
        shape_replot_cache_path,
        shape_replot_pdf_path,
        signal_reject_upper_fraction=(
            shape_replot_signal_reject_upper_fraction
        ),
        pedestal_growth_ratio_max=shape_replot_pedestal_growth_ratio_max,
        pedestal_rms_variation_max=(
            shape_replot_pedestal_rms_variation_max
        ),
        worst_waveforms_to_plot=shape_replot_worst_waveforms_to_plot,
        metric_histogram_bins=shape_replot_metric_histogram_bins,
        metric_qdc_bins=shape_replot_metric_qdc_bins,
        qdc_histogram_bins=shape_replot_qdc_histogram_bins,
        fixed_qdc_range_mV_ns=shape_replot_fixed_qdc_range_mV_ns,
        fixed_qdc_waveforms_to_plot=(
            shape_replot_fixed_qdc_waveforms_to_plot
        ),
        fixed_qdc_population=shape_replot_fixed_qdc_population,
    )
)
print(
    f"Fast shape replot: kept {shape_replot_result['signal_kept']:,}/"
    f"{shape_replot_result['signal_total']:,} signals and "
    f"{shape_replot_result['pedestal_kept']:,}/"
    f"{shape_replot_result['pedestal_total']:,} pedestals."
)
rms_constancy_total = shape_replot_result["pedestal_rms_constancy_total"]
rms_constancy_percent = (
    100.0 * shape_replot_result["pedestal_rms_constancy_kept"]
    / rms_constancy_total if rms_constancy_total else np.nan
)
print(
    f"Local-RMS constancy alone kept "
    f"{shape_replot_result['pedestal_rms_constancy_kept']:,}/"
    f"{rms_constancy_total:,} pedestals ({rms_constancy_percent:.2f}%)."
)
